In [ ]:
# ═══════════════════════════════════════════════════════════════════
# WSDC POINTS PARSER - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

# Core libraries
import pandas as pd
import numpy as np
from datetime import datetime
import math
import re
import json
import traceback
import time
import os
# Web scraping & requests
import urllib3
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup

# Data processing
import us

# Geocoding (optional)
import googlemaps
from geopy.geocoders import Nominatim

# Environment & configuration
from dotenv import load_dotenv
load_dotenv()

# Logging
import logging

# Progress bars
from tqdm.auto import tqdm

# Disable warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
pd.options.mode.chained_assignment = None  # Suppress pandas warnings

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION SETTINGS
# ═══════════════════════════════════════════════════════════════════

# Starting dancer ID (can be overridden from environment)
START_DANCER_ID = int(os.getenv('START_DANCER_ID', '28360'))

# API scraping settings
MAX_ATTEMPTS = int(os.getenv('MAX_ATTEMPTS', '5'))  # Max consecutive failures
REQUEST_TIMEOUT = int(os.getenv('REQUEST_TIMEOUT', '10'))  # API request timeout in seconds
TOKEN_REFRESH_INTERVAL = int(os.getenv('TOKEN_REFRESH_INTERVAL', '120'))  # CSRF token refresh interval

# Delays (in seconds) - Optimized for faster performance
# ИСПРАВЛЕНО: Уменьшена задержка для ускорения парсинга
# REQUEST_DELAY = 0.08 сек дает ~12.5 запросов/сек и время выполнения ~45 минут (вместо 3.5 часов)
# Можно настроить через переменную окружения: REQUEST_DELAY=0.08
REQUEST_DELAY = float(os.getenv('REQUEST_DELAY', '0.08'))  # Delay between requests (уменьшено с 0.2 до 0.08 для ускорения)
BATCH_DELAY = float(os.getenv('BATCH_DELAY', '0'))  # Removed to speed up without server overload

# Google Maps API (optional - only if geocoding is needed)
GOOGLE_MAPS_API_KEY = os.getenv('GOOGLE_MAPS_API_KEY', None)

# Logging setup
os.makedirs('logs', exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/wsdc_parser.log'),
        logging.StreamHandler()
    ]
)

print("✅ Configuration loaded successfully")
print(f"📊 Starting dancer ID: {START_DANCER_ID}")
print(f"⏱️  Request timeout: {REQUEST_TIMEOUT}s")
print(f"🔄 Token refresh interval: {TOKEN_REFRESH_INTERVAL}s")

# ═══════════════════════════════════════════════════════════════════
# DATA PREPROCESSING FUNCTIONS - IMPORT
# ═══════════════════════════════════════════════════════════════════

import sys
from pathlib import Path
import importlib.util

# Определяем директорию, где находится ноутбук и data_preprocessing.py
# Используем абсолютный путь к директории проекта WSDC Points
wsdc_points_dir = Path('/Users/ania/.cursor/projects/tableau/My-Tableau-Projects/WSDC/WSDC Points')

# Если абсолютный путь не работает, пробуем найти относительно текущей директории
if not wsdc_points_dir.exists():
    # Пробуем найти директорию относительно текущей рабочей директории
    current_dir = Path.cwd()
    # Ищем вверх по дереву директорий
    for parent in [current_dir] + list(current_dir.parents):
        potential_dir = parent / 'My-Tableau-Projects' / 'WSDC' / 'WSDC Points'
        if potential_dir.exists():
            wsdc_points_dir = potential_dir
            break
    else:
        # Если не нашли, используем текущую директорию как fallback
        wsdc_points_dir = current_dir

# Добавляем директорию в sys.path для импорта data_preprocessing
if str(wsdc_points_dir) not in sys.path:
    sys.path.insert(0, str(wsdc_points_dir))

try:
    # Пробуем импортировать через importlib для более надежного импорта
    preprocessing_file = wsdc_points_dir / 'data_preprocessing.py'
    
    if preprocessing_file.exists():
        spec = importlib.util.spec_from_file_location("data_preprocessing", preprocessing_file)
        if spec and spec.loader:
            data_preprocessing = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(data_preprocessing)
            
            # Импортируем функции
            normalize_geography = data_preprocessing.normalize_geography
            normalize_dates = data_preprocessing.normalize_dates
            standardize_formats = data_preprocessing.standardize_formats
            validate_data_quality = data_preprocessing.validate_data_quality
            standardize_country = data_preprocessing.standardize_country
            standardize_location = data_preprocessing.standardize_location
            fill_international_state = data_preprocessing.fill_international_state
            validate_coordinates = data_preprocessing.validate_coordinates
            normalize_level = data_preprocessing.normalize_level
            standardize_result = data_preprocessing.standardize_result
            
            print("✅ Data preprocessing functions imported successfully")
            print(f"   Loaded from: {preprocessing_file}")
            DATA_PREPROCESSING_AVAILABLE = True
        else:
            raise ImportError("Could not load data_preprocessing module")
    else:
        raise ImportError(f"File not found: {preprocessing_file}")
        
except Exception as e:
    # Fallback: пробуем обычный импорт
    try:
        from data_preprocessing import (
            normalize_geography,
            normalize_dates,
            standardize_formats,
            validate_data_quality,
            standardize_country,
            standardize_location,
            fill_international_state,
            validate_coordinates,
            normalize_level,
            standardize_result
        )
        print("✅ Data preprocessing functions imported successfully (fallback method)")
        DATA_PREPROCESSING_AVAILABLE = True
    except ImportError as e2:
        print(f"⚠️  Warning: Could not import data preprocessing functions")
        print(f"   Error 1 (importlib): {e}")
        print(f"   Error 2 (direct import): {e2}")
        print(f"   Current working directory: {Path.cwd()}")
        print(f"   WSDC Points directory: {wsdc_points_dir}")
        print(f"   Looking for: {wsdc_points_dir / 'data_preprocessing.py'}")
        print(f"   File exists: {(wsdc_points_dir / 'data_preprocessing.py').exists()}")
        print("   Data quality improvements will be skipped")
        DATA_PREPROCESSING_AVAILABLE = False


✅ Configuration loaded successfully
📊 Starting dancer ID: 26410
⏱️  Request timeout: 10s
🔄 Token refresh interval: 120s
✅ Data preprocessing functions imported successfully
   Loaded from: /Users/ania/.cursor/projects/tableau/My-Tableau-Projects/WSDC/WSDC Points/data_preprocessing.py


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIND MAXIMUM DANCER ID - API METHOD (gap-safe linear scan)
# ═══════════════════════════════════════════════════════════════════

print("\n🚀 Starting maximum dancer ID search using API...")
print("📝 3-phase scan: coarse(500) → fine(100) → linear(+1) — gap-safe, ~60 calls vs ~460\n")

MAX_ID_SAFETY_LIMIT = 100_000
COARSE_STEP = int(os.getenv('MAX_ID_COARSE_STEP', '500'))
FINE_STEP = int(os.getenv('MAX_ID_FINE_STEP', '100'))
MAX_CONSECUTIVE_MISSES = 5
MAX_ID_SCAN_DELAY = float(os.getenv('MAX_ID_SCAN_DELAY', str(min(REQUEST_DELAY, 0.05))))

check_url = 'https://points.worldsdc.com/lookup/autocomplete?q='
check_headers = {
    'Referer': 'https://points.worldsdc.com/lookup2020',
    'User-Agent': 'Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:72.0) Gecko/20100101 Firefox/72.0',
}


def check_id_exists(session, dancer_id):
    """Return True/False if dancer exists; None on network/parse error."""
    try:
        response = session.get(
            check_url + str(dancer_id),
            headers=check_headers,
            verify=False,
            timeout=REQUEST_TIMEOUT,
        )
        data = json.loads(response.text)
        return any(item.get('wscid') == dancer_id for item in data)
    except Exception as e:
        logging.warning(f"Check failed for ID {dancer_id}: {type(e).__name__}: {e}")
        return None


def check_id_exists_retry(session, dancer_id, retries=2):
    """Retry on transient network errors; False if all attempts fail."""
    for attempt in range(retries + 1):
        result = check_id_exists(session, dancer_id)
        if result is not None:
            return result
        if attempt < retries:
            time.sleep(0.5)
    return False


def coarse_scan_upward(session, anchor_id):
    """Jump by COARSE_STEP until first miss; return (last_found, scan_ceiling)."""
    last_found = anchor_id
    current_id = anchor_id + COARSE_STEP
    calls = 1
    scan_ceiling = MAX_ID_SAFETY_LIMIT

    while current_id < MAX_ID_SAFETY_LIMIT:
        if check_id_exists_retry(session, current_id):
            calls += 1
            last_found = current_id
            current_id += COARSE_STEP
            if current_id % 5000 == 0:
                print(f"   Found ID {last_found}, checking higher...")
        else:
            calls += 1
            scan_ceiling = current_id
            print(f"   Coarse upper bound: {scan_ceiling} (last found: {last_found})")
            break
        time.sleep(MAX_ID_SCAN_DELAY)
    else:
        print(f"   Reached safety limit {MAX_ID_SAFETY_LIMIT} (last found: {last_found})")

    return last_found, scan_ceiling, calls


def fine_scan_upward(session, anchor_id, scan_ceiling):
    """Jump by FINE_STEP inside coarse window; gaps are OK (linear tail catches them)."""
    last_found = anchor_id
    current_id = anchor_id + FINE_STEP
    calls = 0

    while current_id < scan_ceiling:
        if check_id_exists_retry(session, current_id):
            calls += 1
            last_found = current_id
            current_id += FINE_STEP
        else:
            calls += 1
            break
        time.sleep(MAX_ID_SCAN_DELAY)

    return last_found, calls


def linear_scan_upward(session, anchor_id):
    """+1 scan with consecutive-miss stop — only the final ~FINE_STEP window."""
    last_found = anchor_id
    dancer_id = anchor_id
    misses = 0
    calls = 0

    while misses < MAX_CONSECUTIVE_MISSES and dancer_id < MAX_ID_SAFETY_LIMIT:
        dancer_id += 1
        if check_id_exists_retry(session, dancer_id):
            calls += 1
            last_found = dancer_id
            misses = 0
        else:
            calls += 1
            misses += 1
        time.sleep(MAX_ID_SCAN_DELAY)

    return last_found, calls


with requests.session() as session:
    starting_id = START_DANCER_ID
    last_found = None
    checks = 0
    checks_fine = 0
    checks_linear = 0

    print(f"🎯 Starting from ID: {starting_id}")
    print(f"   Steps: coarse={COARSE_STEP}, fine={FINE_STEP}, delay={MAX_ID_SCAN_DELAY}s\n")

    try:
        # Phase 1: coarse scan — bracket max within one COARSE_STEP window
        print('1️⃣ Coarse scan...')

        if check_id_exists_retry(session, starting_id):
            checks += 1
            last_found, scan_ceiling, coarse_calls = coarse_scan_upward(session, starting_id)
            checks += coarse_calls
        else:
            checks += 1
            print(f"   ID {starting_id} not found — scanning downward...")
            current_id = starting_id
            anchor = None
            while current_id > 0:
                if check_id_exists_retry(session, current_id):
                    checks += 1
                    anchor = current_id
                    print(f"   Found anchor ID: {anchor}")
                    break
                checks += 1
                current_id -= COARSE_STEP
                time.sleep(MAX_ID_SCAN_DELAY)
            if anchor is None:
                raise ValueError(f"No dancers found at or below ID {starting_id}")
            last_found, scan_ceiling, coarse_calls = coarse_scan_upward(session, anchor)
            checks += coarse_calls

        print()

        # Phase 2: fine scan — skip most IDs, stay gap-safe within coarse window
        print(f'2️⃣ Fine scan (step={FINE_STEP})...')
        last_found, checks_fine = fine_scan_upward(session, last_found, scan_ceiling)
        checks += checks_fine
        print(f"   Fine anchor: {last_found}\n")

        # Phase 3: linear tail — at most ~FINE_STEP + MAX_CONSECUTIVE_MISSES calls
        print(f'3️⃣ Linear tail (+1, stop after {MAX_CONSECUTIVE_MISSES} misses)...')
        last_found, checks_linear = linear_scan_upward(session, last_found)
        checks += checks_linear

        wsdcid_max = last_found

        print(f"\n{'='*60}")
        print("✅ Maximum ID search completed!")
        print("📊 Statistics:")
        print(f"   Total API calls: {checks}")
        print(f"   Fine scan calls: {checks_fine}")
        print(f"   Linear tail calls: {checks_linear}")
        print(f"   Maximum ID found: {wsdcid_max}")
        print(f"{'='*60}\n")

    except KeyboardInterrupt:
        print("\n⚠️  Search interrupted")
        wsdcid_max = last_found if last_found is not None else starting_id - 1
        print(f"🎯 Using best ID found: {wsdcid_max}")

    except Exception as e:
        logging.error(f"Critical error in ID search: {type(e).__name__}: {e}")
        print(f"❌ Critical error: {e}")
        wsdcid_max = last_found if last_found is not None else starting_id - 1
        print(f"⚠️  Using fallback ID: {wsdcid_max}")

    print(f"\n📊 Variable 'wsdcid_max' set to: {wsdcid_max}\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 📁 LOAD DATA FROM LOCAL JSON FILE (alternative to API scraping)
# ═══════════════════════════════════════════════════════════════════
# 
# 🎯 НАЗНАЧЕНИЕ: Загрузка данных из локального файла data.json
#                вместо скачивания через API
#
# ⚠️  ИСПОЛЬЗОВАНИЕ:
#    Запустите эту ячейку ВМЕСТО ячеек 1-3 (Max ID Search, API Scraper, Token Refresh)
#    если у вас уже есть файл data.json с данными
#
# 📋 ТРЕБОВАНИЯ:
#    - Файл data.json должен быть в текущей директории
#    - Формат: JSON массив объектов танцоров
# ═══════════════════════════════════════════════════════════════════

import json
import logging
from datetime import datetime

print("\n" + "="*80)
print("📁 LOADING DATA FROM LOCAL JSON FILE")
print("="*80 + "\n")

try:
    # Проверка наличия файла
    import os
    if not os.path.exists('data.json'):
        raise FileNotFoundError("❌ File 'data.json' not found in current directory!")
    
    file_size = os.path.getsize('data.json') / (1024 * 1024)  # MB
    print(f"📊 File size: {file_size:.1f} MB")
    print(f"⏳ Loading... (this may take a moment)")
    
    start_time = datetime.now()
    
    # Загружаем JSON
    with open('data.json', 'r', encoding='utf-8') as f:
        all_dancer_data = json.load(f)
    
    load_time = (datetime.now() - start_time).total_seconds()
    
    # Статистика
    dancer_count = len(all_dancer_data)
    
    print(f"\n✅ Data loaded successfully!")
    print(f"   📊 Dancers loaded: {dancer_count:,}")
    print(f"   ⏱️  Load time: {load_time:.2f}s")
    print(f"   💾 Memory used: {file_size:.1f} MB")
    
    # Показываем структуру первой записи
    if dancer_count > 0:
        first_dancer = all_dancer_data[0]
        print(f"\n📋 Data structure (first record):")
        print(f"   Keys: {list(first_dancer.keys())}")
        
        if 'leader' in first_dancer:
            leader = first_dancer['leader']
            if 'dancer' in leader:
                dancer = leader['dancer']
                print(f"   Example dancer: {dancer.get('first_name', '')} {dancer.get('last_name', '')} (ID: {dancer.get('id', 'N/A')})")
        
        if 'follower' in first_dancer:
            follower = first_dancer['follower']
            if 'placements' in follower:
                print(f"   Has placements data: Yes")
    
    # Создаем алиас для совместимости с другими ячейками
    soup_list = all_dancer_data
    
    # Логирование
    logging.info(f"Loaded {dancer_count:,} dancers from data.json in {load_time:.2f}s")
    logging.info(f"Created soup_list alias with {len(soup_list):,} dancers")
    
    print(f"\n{'='*80}")
    print(f"✅ READY TO PROCESS")
    print(f"{'='*80}")
    print(f"\n💡 Next steps:")
    print(f"   1. Skip cells 1, 3, 4 (Max ID Search, API Scraper, Token Refresh)")
    print(f"   2. Run Cell 5 (Extract Role Info) and continue sequentially")
    print(f"   3. All subsequent cells will use data from 'soup_list' variable")
    
    print(f"\n✅ Cell 2 (Load from JSON) complete!\n")

except FileNotFoundError as e:
    print(f"\n❌ ERROR: {e}")
    print(f"\n💡 Solution:")
    print(f"   1. Make sure 'data.json' is in the same directory as this notebook")
    print(f"   2. Or run cells 1, 3, 4 to download data from API")
    logging.error(f"data.json not found: {e}")

except json.JSONDecodeError as e:
    print(f"\n❌ ERROR: Invalid JSON format")
    print(f"   Details: {e}")
    print(f"\n💡 Solution:")
    print(f"   1. Check if data.json is a valid JSON file")
    print(f"   2. Re-download data using cells 1, 3, 4")
    logging.error(f"JSON decode error: {e}")

except Exception as e:
    print(f"\n❌ ERROR: {e}")
    print(f"\n💡 If problem persists, use cells 1, 3, 4 to download fresh data from API")
    logging.error(f"Error loading data.json: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# API SCRAPER - BALANCED OPTIMIZATION (50% faster, server-friendly)
# Improvements: Single API call per ID, reduced delays, smart error handling
# ═══════════════════════════════════════════════════════════════════

print("\n🚀 Starting API scraper...")
print(f"   Target range: 1 → {wsdcid_max:,}")
print(f"   ⚡ Optimized mode: REQUEST_DELAY={REQUEST_DELAY}s (balanced speed)")

# Estimate execution time
estimated_seconds = wsdcid_max * REQUEST_DELAY
estimated_hours = estimated_seconds / 3600
estimated_minutes = (estimated_seconds % 3600) / 60
print(f"   ⏱️  Estimated time: ~{int(estimated_hours)}h {int(estimated_minutes)}m (actual may vary)")

# API endpoints and headers
token_url = 'https://points.worldsdc.com/lookup2020'
token_headers = {
    'Referer': 'https://points.worldsdc.com/lookup2020',
    'User-Agent': 'Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:72.0) Gecko/20100101 Firefox/72.0',
}

contester_url = 'https://points.worldsdc.com/lookup2020/find'
contester_headers = {
    'Referer': 'https://points.worldsdc.com/lookup',
    'User-Agent': 'Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:72.0) Gecko/20100101 Firefox/72.0',
    'Content-Type': 'application/x-www-form-urlencoded;charset=UTF-8',
    'Origin': 'https://points.worldsdc.com/lookup2020',
    'X-Requested-With': 'XMLHttpRequest'
}

# Helper functions
def get_token(session):
    """Get CSRF token from the website"""
    try:
        r = session.get(token_url, headers=token_headers, verify=False, timeout=REQUEST_TIMEOUT)
        match = re.search(r'name="_token" value="(.*?)"', r.text)
        if not match:
            raise ValueError("CSRF token not found in HTML")
        token = match.group(1)
        logging.debug(f"New CSRF token obtained: {token[:20]}...")
        return token
    except Exception as e:
        logging.error(f"Error getting token: {type(e).__name__}: {e}")
        raise

def get_contester_info(session, token_csrf, wsdcid, max_retries=3):
    """
    Get dancer information with retry logic.
    
    Parameters:
    -----------
    session : requests.Session
        Active session
    token_csrf : str
        CSRF token
    wsdcid : int
        Dancer ID
    max_retries : int
        Maximum number of retry attempts (default: 3)
        
    Returns:
    --------
    dict
        Dancer data or None on failure
    """
    payload = {
        'num': wsdcid,
        '_token': token_csrf
    }
    
    for attempt in range(max_retries):
        try:
            r = session.post(
                contester_url, 
                data=payload, 
                headers=contester_headers, 
                verify=False, 
                timeout=REQUEST_TIMEOUT
            )
            data = json.loads(r.text)
            return data
        except json.JSONDecodeError as e:
            logging.warning(f"JSON decode error for ID {wsdcid} (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(0.5)
            else:
                logging.error(f"Failed to decode JSON for ID {wsdcid} after {max_retries} attempts")
                return None
        except Exception as e:
            logging.warning(f"Error getting info for ID {wsdcid} (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(0.5)
            else:
                logging.error(f"Failed to get info for ID {wsdcid} after {max_retries} attempts")
                return None
    
    return None

# Initialize session and data
with requests.session() as s:
    soup_list = []

    # Statistics tracking
    # ИСПРАВЛЕНО: total должен быть wsdcid_max, так как мы обрабатываем ID от 1 до wsdcid_max включительно
    # Это означает, что всего будет обработано wsdcid_max ID (от 1 до wsdcid_max)
    stats = {
        'total': wsdcid_max,  # Всего ID для обработки (от 1 до wsdcid_max включительно)
        'found': 0,
        'not_found': 0,
        'errors': 0
    }

    # Token management
    token_csrf = None
    last_token_time = 0

    print(f"\n{'='*60}")
    print("🔄 Scraping dancer data...")
    print(f"{'='*60}\n")

    # Track execution time
    start_time = time.time()

    # Main loop with progress bar
    try:
        # ИСПРАВЛЕНО: range(1, wsdcid_max + 1) чтобы включить последний ID
        # range(1, wsdcid_max) создает последовательность от 1 до wsdcid_max-1
        # Например, если wsdcid_max = 27124, то range(1, 27124) = [1, 2, ..., 27123]
        # Поэтому нужно range(1, wsdcid_max + 1) чтобы включить ID 27124
        pbar = tqdm(range(1, wsdcid_max + 1), desc="Scraping", unit="ID")
    
        for wsdcid in pbar:
            try:
                # Smart token refresh
                if token_csrf is None or time.time() - last_token_time > TOKEN_REFRESH_INTERVAL:
                    token_csrf = get_token(s)
                    last_token_time = time.time()
                    logging.info(f"CSRF token refreshed at ID {wsdcid}")
            
                # Get dancer information directly (no pre-check needed - saves 50% API calls)
                contester_data = get_contester_info(s, token_csrf, wsdcid)
            
                # Process response
                if contester_data and contester_data != [] and contester_data != {}:
                    soup_list.append(contester_data)
                    stats['found'] += 1
                elif contester_data is None:
                    stats['errors'] += 1
                else:
                    stats['not_found'] += 1
            
                # Update progress bar with statistics
                pbar.set_postfix({
                    'found': stats['found'],
                    'errors': stats['errors']
                })
            
                # Balanced delay - not too aggressive, not too slow
                time.sleep(REQUEST_DELAY)
            
                # Batch delay (only if configured)
                if BATCH_DELAY > 0 and wsdcid % 10 == 0:
                    time.sleep(BATCH_DELAY)
            
            except KeyboardInterrupt:
                print(f"\n\n⚠️  Interrupted by user at ID {wsdcid}")
                break
            except Exception as e:
                stats['errors'] += 1
                logging.error(f"Error processing ID {wsdcid}: {type(e).__name__}: {e}")
                time.sleep(2)  # Wait longer on errors
    
        pbar.close()

    except KeyboardInterrupt:
        print(f"\n\n⚠️  Scraping interrupted by user")
    except Exception as e:
        logging.error(f"Critical error in main loop: {type(e).__name__}: {e}")
        traceback.print_exc()

    # Final statistics
    end_time = time.time()
    elapsed_time = end_time - start_time
    elapsed_hours = int(elapsed_time // 3600)
    elapsed_minutes = int((elapsed_time % 3600) // 60)
    elapsed_seconds = int(elapsed_time % 60)

    print(f"\n{'='*60}")
    print("✅ Scraping completed!")
    print(f"{'='*60}")
    print(f"📊 Statistics:")
    print(f"   Total processed: {stats['found'] + stats['not_found'] + stats['errors']:,}")
    print(f"   ✅ Found: {stats['found']:,} ({stats['found']/(stats['found']+stats['not_found']+stats['errors'])*100:.1f}%)")
    print(f"   ⭕ Not found: {stats['not_found']:,} ({stats['not_found']/(stats['found']+stats['not_found']+stats['errors'])*100:.1f}%)")
    print(f"   ❌ Errors: {stats['errors']:,} ({stats['errors']/(stats['found']+stats['not_found']+stats['errors'])*100:.1f}%)")
    print(f"⏱️  Execution time: {elapsed_hours}h {elapsed_minutes}m {elapsed_seconds}s")
    print(f"📦 Data saved to soup_list ({len(soup_list):,} records)")
    print(f"{'='*60}\n")

    logging.info(f"Scraping complete: {stats['found']} found, {stats['not_found']} not found, {stats['errors']} errors. Time: {elapsed_hours}h {elapsed_minutes}m {elapsed_seconds}s")

    print(f"✅ Cell 3 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SAVE DATA TO JSON - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

import os
from pathlib import Path

print("\n💾 Saving scraped data to JSON file...")

# Проверка наличия данных
if 'soup_list' not in globals() or not soup_list:
    print("⚠️  Warning: soup_list is empty or not found!")
    print("   Please run Cell 3 first to scrape data")
    logging.warning("soup_list not available for saving")
else:
    # Конфигурация
    output_file = "data.json"
    
    try:
        # Статистика перед сохранением
        num_records = len(soup_list)
        
        print(f"   📊 Records to save: {num_records:,}")
        
        # Сохранение с оптимизированными параметрами
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(
                soup_list, 
                f, 
                ensure_ascii=False,  # Сохраняем Unicode символы как есть
                indent=2             # Красивое форматирование
            )
        
        # Получаем размер файла
        file_size = Path(output_file).stat().st_size
        file_size_mb = file_size / 1024 / 1024
        
        # Успешное сохранение
        print(f"\n✅ Data saved successfully!")
        print(f"   📁 File: {output_file}")
        print(f"   📊 Records: {num_records:,}")
        print(f"   💾 Size: {file_size:,} bytes ({file_size_mb:.2f} MB)")
        
        # Проверка целостности (опционально - загружаем обратно)
        try:
            with open(output_file, "r", encoding="utf-8") as f:
                test_load = json.load(f)
            
            if len(test_load) == num_records:
                print(f"   ✅ File integrity verified ({len(test_load):,} records)")
            else:
                print(f"   ⚠️  Warning: Record count mismatch!")
                print(f"      Expected: {num_records:,}, Got: {len(test_load):,}")
        except Exception as e:
            print(f"   ⚠️  Could not verify file integrity: {e}")
        
        logging.info(f"Data saved to {output_file}: {num_records:,} records, {file_size_mb:.2f} MB")
        
    except PermissionError:
        print(f"❌ Error: Permission denied to write {output_file}")
        logging.error(f"Permission denied when saving to {output_file}")
    except Exception as e:
        print(f"❌ Error saving data: {type(e).__name__}: {e}")
        logging.error(f"Error saving data to JSON: {type(e).__name__}: {e}")

print(f"\n✅ Cell 4 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EXTRACT ROLE INFORMATION - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

def extract_role_info(dancer_data):
    """
    Extract role information from dancer data dictionary.
    
    Parameters:
    -----------
    dancer_data : dict
        Dictionary containing dancer information from API
        
    Returns:
    --------
    dict
        Dictionary with extracted role information
    """
    # Извлечение имени танцора с очисткой
    dancer_first = dancer_data.get('dancer_first', '').replace('\t', '').strip()
    dancer_last = dancer_data.get('dancer_last', '').replace('\t', '').strip()

    # Безопасное извлечение non_dominate_lookup
    non_dom_lookup = dancer_data.get('non_dominate_lookup', [])
    if non_dom_lookup and isinstance(non_dom_lookup, list) and len(non_dom_lookup) > 0:
        non_dom = non_dom_lookup[0]
        non_dominate_required = non_dom.get('non_dominate_required', '')
        non_dominate_allowed = non_dom.get('non_dominate_allowed', '')
        non_dominate_recommended = non_dom.get('non_dominate_recommended', '')
    else:
        non_dominate_required = ''
        non_dominate_allowed = ''
        non_dominate_recommended = ''

    roles = {
        'dancer_id': dancer_data.get('dancer_wsdcid', ''),
        'dancer_name': f"{dancer_first} {dancer_last}".strip(),
        'dominate_role': dancer_data.get('short_dominate_role', ''),
        'dominate_required': dancer_data.get('dominate_required', ''),
        'dominate_allowed': dancer_data.get('dominate_allowed', ''),
        'non_dominate_role': dancer_data.get('short_non_dominate_role', ''),
        'non_dominate_required': non_dominate_required,
        'non_dominate_allowed': non_dominate_allowed,
        'non_dominate_recommended': non_dominate_recommended,
        'non_dominate_role_highest_level_points': dancer_data.get('non_dominate_role_highest_level_points', ''),
        'non_dominate_role_highest_level': dancer_data.get('non_dominate_role_highest_level', ''),
    }
    return roles

def remove_duplicates_and_sort(filename='empty_dancer_ids.txt'):
    """
    Удаляет дубликаты и сортирует ID танцоров в файле.
    
    Parameters:
    -----------
    filename : str
        Имя файла с ID танцоров (default: 'empty_dancer_ids.txt')
        
    Returns:
    --------
    int
        Количество уникальных ID после обработки
    """
    import os
    
    # Проверка существования файла
    if not os.path.exists(filename):
        logging.warning(f"File '{filename}' not found. Creating empty file.")
        with open(filename, 'w') as f:
            pass
        return 0
    
    try:
        # Чтение уникальных ID
        with open(filename, 'r') as f:
            unique_ids = sorted(set(int(line.strip()) for line in f if line.strip()))
        
        # Запись обратно
        with open(filename, 'w') as f:
            for dancer_id in unique_ids:
                f.write(f"{dancer_id}\n")
        
        logging.info(f"File '{filename}' processed: {len(unique_ids)} unique IDs")
        return len(unique_ids)
        
    except Exception as e:
        logging.error(f"Error processing '{filename}': {type(e).__name__}: {e}")
        return 0

print("\n🔄 Processing role information...")

# Инициализация
all_dancers_info = []
errors = 0
error_details = []

print(f"   Processing {len(soup_list):,} dancers...\n")

# Progress bar с обработкой ошибок
for dancer_data in tqdm(soup_list, desc="Extracting roles", unit="dancer"):
    try:
        dancer_role_info = extract_role_info(dancer_data)
        all_dancers_info.append(dancer_role_info)
    except Exception as e:
        errors += 1
        dancer_id = dancer_data.get('dancer_wsdcid', 'Unknown')
        error_msg = f"ID {dancer_id}: {type(e).__name__}: {e}"
        error_details.append(error_msg)
        logging.error(f"Error extracting role info: {error_msg}")

# Создание DataFrame
dancer_role_info_df = pd.DataFrame(all_dancers_info)

print(f"\n✅ Role extraction complete!")
print(f"   📊 Records extracted: {len(all_dancers_info):,}")
print(f"   ❌ Errors: {errors}")

if errors > 0:
    print(f"   ⚠️  First 5 errors:")
    for detail in error_details[:5]:
        print(f"      - {detail}")

# Проверка пустых имен (улучшенная)
empty_mask = dancer_role_info_df['dancer_name'].isna() | \
             (dancer_role_info_df['dancer_name'] == '') | \
             (dancer_role_info_df['dancer_name'].str.strip() == '')
empty_count = empty_mask.sum()

print(f"\n📋 DataFrame info:")
print(f"   Shape: {dancer_role_info_df.shape}")
print(f"   Empty names: {empty_count:,}")

if empty_count > 0:
    empty_ids = dancer_role_info_df[empty_mask]['dancer_id'].tolist()
    print(f"   Saving {len(empty_ids)} empty IDs to 'empty_dancer_ids.txt'...")
    
    # Сохранение empty IDs
    with open('empty_dancer_ids.txt', 'w') as f:
        for dancer_id in empty_ids:
            f.write(f"{dancer_id}\n")
    
    # Удаление дубликатов и сортировка
    unique_count = remove_duplicates_and_sort('empty_dancer_ids.txt')
    print(f"   ✓ Saved {unique_count} unique IDs")
    
    # Удаление записей с пустыми именами
    dancer_role_info_df = dancer_role_info_df[~empty_mask].reset_index(drop=True)
    print(f"   ✓ Removed {empty_count:,} records with empty names")
    print(f"   ✓ New shape: {dancer_role_info_df.shape}")

# Применение replacement_dict (если доступен)
if 'replacement_dict' in globals():
    print(f"\n🔄 Normalizing values using replacement_dict...")
    for col in dancer_role_info_df.columns:
        if col in replacement_dict:
            dancer_role_info_df[col] = dancer_role_info_df[col].replace(replacement_dict[col])
    print(f"   ✓ Values normalized")
else:
    print(f"\n⚠️  replacement_dict not found - skipping normalization")

# Финальная статистика
print(f"\n📊 Final statistics:")
print(f"   Total dancers: {len(dancer_role_info_df):,}")
print(f"   Unique dancer IDs: {dancer_role_info_df['dancer_id'].nunique():,}")
print(f"   Dominate roles: {dancer_role_info_df['dominate_role'].nunique()} unique")
print(f"   Non-dominate roles: {dancer_role_info_df['non_dominate_role'].nunique()} unique")

logging.info(f"Role extraction: {len(dancer_role_info_df):,} dancers extracted, {errors} errors")

# Создаем алиас для совместимости с другими ячейками
dancer_role_info = dancer_role_info_df

print(f"\n✅ Cell 5 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# KEY VALIDATION - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

# Required keys for validation
required_keys = [
    'dancer_wsdcid',
    'dancer_first',
    'dancer_last',
    'short_dominate_role',
    'dominate_required',
    'dominate_allowed',
    'short_non_dominate_role',
    'non_dominate_required',
    'non_dominate_allowed',
    'non_dominate_recommended',
    'non_dominate_role_highest_level_points', 
    'non_dominate_role_highest_level'
]

lookup_keys = ['non_dominate_required', 'non_dominate_allowed', 'non_dominate_recommended']

def check_keys_in_soup_list(soup_list, show_errors=True, max_errors_to_show=10, test_mode=False):
    """
    Проверка наличия всех необходимых ключей в soup_list.
    
    Parameters:
    -----------
    soup_list : list
        Список словарей с данными танцоров
    show_errors : bool
        Показывать ли детали ошибок (default: True)
    max_errors_to_show : int
        Максимальное количество ошибок для вывода (default: 10)
    test_mode : bool
        Если True, проверяет только первые 100 записей (default: False)
        
    Returns:
    --------
    dict
        Словарь со статистикой валидации
    """
    if test_mode:
        soup_list = soup_list[:100]
    
    print(f"\n🔍 Validating keys in soup_list...")
    print(f"   Total records: {len(soup_list):,}")
    
    stats = {
        'total': len(soup_list),
        'valid': 0,
        'invalid': 0,
        'missing_keys_count': {},
        'errors': []
    }
    
    # Progress bar
    pbar = tqdm(soup_list, desc="Validating", unit="record")
    
    for i, dancer_data in enumerate(pbar):
        missing_keys = []
        dancer_id = dancer_data.get('dancer_wsdcid', f'index_{i}')
        
        # Получаем non_dominate_lookup если есть
        non_dom_lookup = None
        if 'non_dominate_lookup' in dancer_data:
            lookup_data = dancer_data['non_dominate_lookup']
            if isinstance(lookup_data, list) and len(lookup_data) > 0:
                if isinstance(lookup_data[0], dict):
                    non_dom_lookup = lookup_data[0]
        
        # Проверяем каждый ключ
        for key in required_keys:
            # Прямая проверка в dancer_data
            if key in dancer_data:
                continue
            
            # Проверка в non_dominate_lookup[0]
            if key in lookup_keys and non_dom_lookup:
                if key in non_dom_lookup:
                    continue
            
            # Ключ отсутствует
            missing_keys.append(key)
            # Счетчик по ключам
            stats['missing_keys_count'][key] = stats['missing_keys_count'].get(key, 0) + 1
        
        # Обновляем статистику
        if missing_keys:
            stats['invalid'] += 1
            error_info = {
                'index': i,
                'dancer_id': dancer_id,
                'missing_keys': missing_keys
            }
            stats['errors'].append(error_info)
        else:
            stats['valid'] += 1
        
        # Обновляем progress bar
        pbar.set_postfix({
            'valid': stats['valid'],
            'invalid': stats['invalid']
        })
    
    pbar.close()
    
    # Выводим результаты
    print(f"\n{'='*60}")
    print(f"✅ VALIDATION COMPLETE")
    print(f"{'='*60}")
    print(f"📊 Statistics:")
    print(f"   Total records: {stats['total']:,}")
    print(f"   ✅ Valid: {stats['valid']:,} ({stats['valid']/stats['total']*100:.1f}%)")
    print(f"   ❌ Invalid: {stats['invalid']:,} ({stats['invalid']/stats['total']*100:.1f}%)")
    
    # Статистика по отсутствующим ключам
    if stats['missing_keys_count']:
        print(f"\n📋 Missing keys frequency:")
        sorted_keys = sorted(stats['missing_keys_count'].items(), key=lambda x: x[1], reverse=True)
        for key, count in sorted_keys:
            print(f"   {key}: {count:,} times ({count/stats['total']*100:.1f}%)")
    
    # Детали ошибок (если нужно)
    if show_errors and stats['errors']:
        print(f"\n❌ Sample errors (showing first {min(max_errors_to_show, len(stats['errors']))}):")
        for error in stats['errors'][:max_errors_to_show]:
            print(f"   Index {error['index']}, ID {error['dancer_id']}: missing {error['missing_keys']}")
        
        if len(stats['errors']) > max_errors_to_show:
            print(f"   ... and {len(stats['errors']) - max_errors_to_show} more errors")
    
    # Логирование
    logging.info(f"Key validation: {stats['valid']}/{stats['total']} valid ({stats['valid']/stats['total']*100:.1f}%)")
    if stats['invalid'] > 0:
        logging.warning(f"Found {stats['invalid']} records with missing keys")
    
    return stats

# Run validation
validation_stats = check_keys_in_soup_list(soup_list, show_errors=True, max_errors_to_show=15)

print(f"\n✅ Cell 8 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EXTRACT DANCER INFO - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

def extract_dancer_info(soup_list):
    """
    Извлекает информацию о танцорах, ролях, танцах и баллах.
    
    Parameters:
    -----------
    soup_list : list
        Список словарей с данными танцоров от API
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame с извлеченной информацией
    """
    dancer_info = []
    errors = 0
    processed_dancers = 0
    
    print(f"\n🔄 Extracting dancer info from {len(soup_list):,} records...")
    
    # Progress bar
    pbar = tqdm(soup_list, desc="Extracting", unit="dancer")
    
    for soup in pbar:
        try:
            # ВАЖНО: Используем dancer_wsdcid - это и есть WSDC ID танцора!
            dancer_id = soup.get('dancer_wsdcid', '')
            
            for role, role_data in soup.items():
                if role in ['leader', 'follower'] and isinstance(role_data, dict):
                    # Получаем required уровень, если он есть
                    level_required = role_data.get('level', {}).get('required', '')
                    
                    placements = role_data.get('placements', {})
                    if isinstance(placements, dict):
                        for dance, dance_data in placements.items():
                            if isinstance(dance_data, dict):
                                for level, level_data in dance_data.items():
                                    if isinstance(level_data, dict):
                                        total_points = level_data.get('total_points', None)
                                        dancer_info.append([dancer_id, role, dance, level, total_points])
            
            processed_dancers += 1
            
            # Обновляем progress bar
            if processed_dancers % 100 == 0:
                pbar.set_postfix({
                    'records': len(dancer_info),
                    'errors': errors
                })
                
        except Exception as e:
            errors += 1
            dancer_id = soup.get('dancer_wsdcid', 'Unknown')
            logging.error(f"Error extracting dancer info for ID {dancer_id}: {type(e).__name__}: {e}")
    
    pbar.close()
    
    # Создаем DataFrame
    columns = ['dancer_id', 'role', 'dance', 'level', 'total_points']
    
    if not dancer_info:
        print("⚠️  No data extracted!")
        dancer_df = pd.DataFrame(columns=columns)
    else:
        dancer_df = pd.DataFrame(dancer_info, columns=columns)
    
    # Статистика
    print(f"\n✅ Extraction complete!")
    print(f"   📊 Records extracted: {len(dancer_info):,}")
    print(f"   👥 Dancers processed: {processed_dancers:,}")
    print(f"   ❌ Errors: {errors}")
    if len(dancer_df) > 0:
        print(f"   📋 DataFrame shape: {dancer_df.shape}")
        print(f"   💯 Unique dancers: {dancer_df['dancer_id'].nunique():,}")
        print(f"   🎭 Unique roles: {dancer_df['role'].nunique()}")
        print(f"   💃 Unique dances: {dancer_df['dance'].nunique()}")
    
    logging.info(f"Extracted {len(dancer_info):,} records from {processed_dancers:,} dancers")
    
    return dancer_df

# Extract info
dancers_points_info = extract_dancer_info(soup_list)

# Нормализация значений через replacement_dict (если доступен)
print(f"\n✅ Cell 7 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EXTRACT DANCER RESULTS - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

def extract_dancer_results(soup_list):
    """
    Извлекает детальную информацию о соревнованиях танцоров.
    
    Parameters:
    -----------
    soup_list : list
        Список словарей с данными танцоров от API
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame с детальными результатами соревнований
    """
    dancer_results = []
    errors = 0
    processed_dancers = 0
    
    print(f"\n🔄 Extracting dancer results from {len(soup_list):,} records...")
    
    # Progress bar
    pbar = tqdm(soup_list, desc="Extracting", unit="dancer")
    
    for soup in pbar:
        try:
            # ВАЖНО: Используем dancer_wsdcid - это и есть WSDC ID танцора!
            dancer_id = soup.get('dancer_wsdcid', '')
            
            for event_role, role_data in soup.items():
                if event_role in ['leader', 'follower'] and isinstance(role_data, dict):
                    placements = role_data.get('placements', {})
                    
                    # Проверяем, является ли placements словарем
                    if isinstance(placements, dict):
                        for dance, dance_data in placements.items():
                            if isinstance(dance_data, dict):
                                event_dance = dance
                                for level, level_data in dance_data.items():
                                    if isinstance(level_data, dict):
                                        # Безопасный доступ к division
                                        division = level_data.get('division', {})
                                        if isinstance(division, dict):
                                            event_competition = division.get('name', '')
                                        else:
                                            event_competition = ''
                                        
                                        # Обработка competitions
                                        competitions = level_data.get('competitions', [])
                                        if isinstance(competitions, list):
                                            for competition in competitions:
                                                if isinstance(competition, dict):
                                                    event_data = competition.get('event', {})
                                                    if isinstance(event_data, dict):
                                                        event_location = event_data.get('location', '')
                                                        event_name_id = event_data.get('id', '')
                                                        event_name = event_data.get('name', '').strip()
                                                        event_date = event_data.get('date', '')
                                                    else:
                                                        event_location = event_name_id = event_name = event_date = ''
                                                    
                                                    event_result = competition.get('result', '')
                                                    event_points = competition.get('points', '')
                                                    
                                                    row_data = [
                                                        dancer_id, event_dance, event_competition, 
                                                        event_role, event_location, event_name_id, 
                                                        event_name, event_date, event_result, event_points
                                                    ]
                                                    dancer_results.append(row_data)
            
            processed_dancers += 1
            
            # Обновляем progress bar
            if processed_dancers % 100 == 0:
                pbar.set_postfix({
                    'records': len(dancer_results),
                    'errors': errors
                })
                
        except Exception as e:
            errors += 1
            dancer_id = soup.get('dancer_wsdcid', 'Unknown')
            logging.error(f"Error extracting dancer results for ID {dancer_id}: {type(e).__name__}: {e}")
    
    pbar.close()
    
    # Создаем DataFrame
    columns = [
        'dancer_id', 'event_dance', 'event_competition', 'event_role', 
        'event_location', 'event_name_id', 'event_name', 'event_date', 
        'event_result', 'event_points'
    ]
    
    if not dancer_results:
        print("⚠️  No data extracted!")
        dancer_results_df = pd.DataFrame(columns=columns)
    else:
        dancer_results_df = pd.DataFrame(dancer_results, columns=columns)
    
    # Статистика
    print(f"\n✅ Extraction complete!")
    print(f"   📊 Records extracted: {len(dancer_results):,}")
    print(f"   👥 Dancers processed: {processed_dancers:,}")
    print(f"   ❌ Errors: {errors}")
    if len(dancer_results_df) > 0:
        print(f"   📋 DataFrame shape: {dancer_results_df.shape}")
        print(f"   💯 Unique dancers: {dancer_results_df['dancer_id'].nunique():,}")
        print(f"   🎭 Unique roles: {dancer_results_df['event_role'].nunique()}")
        print(f"   💃 Unique dances: {dancer_results_df['event_dance'].nunique()}")
        print(f"   🏆 Unique events: {dancer_results_df['event_name_id'].nunique():,}")
    
    logging.info(f"Extracted {len(dancer_results):,} competition records from {processed_dancers:,} dancers")
    
    return dancer_results_df

# Extract results
dancers_results_info = extract_dancer_results(soup_list)

# Нормализация значений через replacement_dict (если доступен)
print(f"\n✅ Cell 8 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LOCATION NORMALIZATION - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("🔄 Normalizing event locations... Which we have done. This cell is now optimized...")

if 'dancers_results_info' not in globals():
    print(f"⚠️  Warning: dancers_results_info not found!")
    logging.warning(f"dancers_results_info not available")
elif not isinstance(dancers_results_info, pd.DataFrame):
    print(f"⚠️  Warning: dancers_results_info is not a DataFrame (type: {type(dancers_results_info).__name__})")
    logging.warning(f"dancers_results_info has wrong type: {type(dancers_results_info)}")
elif dancers_results_info.empty:
    print(f"⚠️  Warning: dancers_results_info DataFrame is empty!")
    logging.warning(f"dancers_results_info DataFrame is empty")
else:
    initial_count = len(dancers_results_info)
    
    exact_location_mapping = {
        'Adelaide, South Australia, Australia': 'Adelaide, Australia',
        'Budapest': 'Budapest, Hungary',
        'Calgar Yy, Alberta': 'Calgary, Canada',
        'Czech Republic': 'Brno, Czech Republic',
        'Dallas, Texas': 'Dallas, TX',
        'East Rutherford': 'East Rutherford, NJ',
        'Edmonton, ON': 'Edmonton, Canada',
        'Gold Coast, Queensland': 'Gold Coast, Australia',
        'Israel': 'Tel Aviv, Israel',
        'Ottawa': 'Ottawa, Canada',
        'Paris': 'Paris, France',
        'Sweden': 'Stockholm, Sweden',
        'Toulouse': 'Toulouse, France',
        'Redmond, Oregon': 'Redmond, OR',
        'Seoul, South Korea': 'Seoul, Republic of Korea',
        'Seoul, Korea': 'Seoul, Republic of Korea',
        'Concord CA': 'Concord, CA',
    }
    
    event_name_mapping = {
        'Go West Swing Fest': 'Fremantle, Australia',
        'BeeMAD': 'Madrid, Spain',
    }
    
    partial_replacements = {
        'Scotland': 'United Kingdom',
        'ENGLAND': 'United Kingdom',
        'England': 'United Kingdom',
        'UK': 'United Kingdom',
        'FRANCE': 'France',
        'QC Canada': 'Canada',
        'QC': 'Canada',
        'Isreal': 'Israel',
        'Washington Dc': 'Washington',
        'Kindom': 'Kingdom',
        'Italia': 'Italy',
        'BC': 'Canada',
        'Bernadino': 'Bernardino',
        'Minn / St. Paul': 'St. Paul',
        'St. Burlatskaya, Russia': 'Samara, Russia',
    }
    
    changes_count = {'exact': 0, 'event_name': 0, 'partial': {}}
    
    # 1. Exact matches
    if 'event_location' in dancers_results_info.columns:
        for old_value, new_value in exact_location_mapping.items():
            mask = dancers_results_info['event_location'] == old_value
            count = mask.sum()
            if count > 0:
                dancers_results_info.loc[mask, 'event_location'] = new_value
                changes_count['exact'] += count
                logging.info(f"Location normalization (exact): '{old_value}' → '{new_value}': {count:,} records")
    
    # 2. Event-specific
    if 'event_name' in dancers_results_info.columns and 'event_location' in dancers_results_info.columns:
        for event_name, location in event_name_mapping.items():
            mask = dancers_results_info['event_name'] == event_name
            count = mask.sum()
            if count > 0:
                dancers_results_info.loc[mask, 'event_location'] = location
                changes_count['event_name'] += count
                logging.info(f"Location normalization (event-specific): '{event_name}' → '{location}': {count:,} records")
    
    # 3. Partial replacements
    if 'event_location' in dancers_results_info.columns:
        for old_part, new_part in partial_replacements.items():
            before = dancers_results_info['event_location'].copy()
            dancers_results_info['event_location'] = dancers_results_info['event_location'].str.replace(old_part, new_part, regex=False)
            count = (before != dancers_results_info['event_location']).sum()
            if count > 0:
                changes_count['partial'][old_part] = count
                logging.info(f"Location normalization (partial): '{old_part}' → '{new_part}': {count:,} records")
    
    total_changes = changes_count['exact'] + changes_count['event_name'] + sum(changes_count['partial'].values())
    
    print(f"✅ Location normalization complete!")
    print(f"   📊 Total records: {initial_count:,}")
    print(f"   🔄 Total changes: {total_changes:,}")
    print(f"   ✓ Exact matches replaced: {changes_count['exact']:,}")
    print(f"   ✓ Event-specific replaced: {changes_count['event_name']:,}")
    print(f"   ✓ Partial replacements: {sum(changes_count['partial'].values()):,}")
    
    if changes_count['partial']:
        print(f"📋 Top partial replacements:")
        sorted_partial = sorted(changes_count['partial'].items(), key=lambda x: x[1], reverse=True)
        for old_val, count in sorted_partial[:10]:
            print(f"      '{old_val}': {count:,} records")
    
    logging.info(f"Location normalization: {total_changes:,} total changes applied")

print(f"\n✅ Cell 9 complete!")



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CREATE LOCATION INFO DATAFRAME
# ═══════════════════════════════════════════════════════════════════

print("📋 Creating location info DataFrame...")

from pathlib import Path

location_info = None
source_messages = []

# 1) Reuse existing in-memory DataFrame if оно уже построено ранее
if 'location_info' in globals():
    existing = globals().get('location_info')
    if isinstance(existing, pd.DataFrame) and not existing.empty:
        location_info = existing.copy()
        source_messages.append("memory (existing location_info)")

# 2) Попытка построить из dancers_results_info
if location_info is None:
    if 'dancers_results_info' not in globals():
        logging.warning("dancers_results_info not available for location derivation")
    else:
        df_results = globals().get('dancers_results_info')
        if isinstance(df_results, pd.DataFrame) and not df_results.empty:
            if 'event_location' in df_results.columns:
                unique_values = df_results['event_location'].dropna().unique()
                location_info = pd.DataFrame({'event_location': unique_values})
                source_messages.append("dancers_results_info[event_location]")
            else:
                logging.warning("dancers_results_info missing 'event_location' column")
        else:
            logging.warning("dancers_results_info is empty or invalid for location derivation")

# 3) Попытка построить на основе events_wsdc (если уже извлекли события)
if location_info is None:
    if 'events_wsdc' in globals():
        df_events = globals().get('events_wsdc')
        if isinstance(df_events, pd.DataFrame) and not df_events.empty and 'location' in df_events.columns:
            unique_values = df_events['location'].dropna().unique()
            location_info = pd.DataFrame({'event_location': unique_values})
            source_messages.append("events_wsdc[location]")

# 4) Фоллбэк: загрузка из существующего CSV (если есть)
if location_info is None:
    csv_path = Path('location_info.csv')
    if csv_path.exists():
        try:
            location_info = pd.read_csv(csv_path)
            source_messages.append("location_info.csv (existing file)")
        except Exception as exc:
            logging.error(f"Failed to read location_info.csv: {exc}")

# 5) Если так и не смогли построить — создаём пустой шаблон
if location_info is None:
    location_info = pd.DataFrame(columns=['event_location'])
    print("⚠️  Unable to derive locations from available sources. Created empty DataFrame.")
    logging.warning("location_info could not be constructed; created empty frame")
else:
    unique_count = len(location_info)
    print("✅ Location info DataFrame ready!")
    if source_messages:
        print(f"   📦 Source: {', '.join(source_messages)}")
    print(f"   📊 Unique locations: {unique_count:,}")
    logging.info(f"location_info prepared with {unique_count:,} entries from {source_messages}")

print(f"✅ Cell 10 complete!")



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PARSE EVENT_LOCATION INTO COMPONENTS
# ═══════════════════════════════════════════════════════════════════
# 
# 🎯 НАЗНАЧЕНИЕ: Парсинг event_location на city, state, country
#
# 📋 СОЗДАЕТ КОЛОНКИ:
#    • event_city - город
#    • event_state - штат/провинция (может быть пустым)
#    • event_country - страна
# ═══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import logging

print("\n🔄 Parsing event_location into components...")

if 'location_info' not in globals():
    print("⚠️  Warning: location_info not found!")
    logging.warning("location_info not available")
elif not isinstance(location_info, pd.DataFrame):
    print(f"⚠️  Warning: location_info is not a DataFrame (type: {type(location_info).__name__})!")
    logging.warning(f"location_info has wrong type: {type(location_info)}")
elif location_info.empty:
    print("⚠️  Warning: location_info DataFrame is empty!")
    logging.warning("location_info DataFrame is empty")
else:
    if 'event_location' not in location_info.columns:
        print("⚠️  Warning: 'event_location' column not found!")
        logging.warning("event_location column not found")
    else:
        initial_count = len(location_info)
        print(f"   Processing {initial_count:,} locations...")
        
        # Инициализируем колонки
        location_info['event_city'] = np.nan
        location_info['event_state'] = np.nan
        location_info['event_country'] = np.nan
        
        # Парсим каждую локацию
        for idx, row in location_info.iterrows():
            location = row['event_location']
            
            if pd.notna(location) and location:
                # Разбиваем по запятой и пробелу
                parts = [p.strip() for p in str(location).split(',')]
                
                # ═══════════════════════════════════════════════════════════
                # УЛУЧШЕННАЯ ЛОГИКА ПАРСИНГА:
                # ═══════════════════════════════════════════════════════════
                # ПРАВИЛО: Последняя часть - это ВСЕГДА страна!
                # 
                # Примеры:
                # "Dallas, TX, United States" → city=Dallas, state=TX, country=United States
                # "Dallas, TX" → city=Dallas, state=TX, country=United States (предполагаем)
                # "Toronto, Canada" → city=Toronto, state='', country=Canada
                # "Dundalk, Co, Louth, Ireland" → city=Dundalk, state='', country=Ireland
                # "Paris, France" → city=Paris, state='', country=France
                # ═══════════════════════════════════════════════════════════
                
                num_parts = len(parts)
                
                if num_parts == 1:
                    # 1 часть: только город
                    location_info.at[idx, 'event_city'] = parts[0]
                
                elif num_parts == 2:
                    # 2 части: City, State/Country
                    location_info.at[idx, 'event_city'] = parts[0]
                    
                    # Если 2 буквы в верхнем регистре - скорее всего штат США
                    if len(parts[1]) == 2 and parts[1].isupper():
                        location_info.at[idx, 'event_state'] = parts[1]
                        location_info.at[idx, 'event_country'] = 'United States'
                    else:
                        # Иначе - это страна
                        location_info.at[idx, 'event_country'] = parts[1]
                
                elif num_parts >= 3:
                    # 3+ части: ПОСЛЕДНЯЯ часть - это СТРАНА!
                    location_info.at[idx, 'event_city'] = parts[0]
                    location_info.at[idx, 'event_country'] = parts[-1]  # ПОСЛЕДНЯЯ часть!
                    
                    # Если страна = United States/USA, то предпоследняя часть - штат
                    country = parts[-1].lower()
                    if 'united states' in country or country == 'usa':
                        # Берем часть перед страной (обычно это штат)
                        if num_parts >= 3:
                            location_info.at[idx, 'event_state'] = parts[-2]
        
        # Статистика
        cities = location_info['event_city'].notna().sum()
        states = location_info['event_state'].notna().sum()
        countries = location_info['event_country'].notna().sum()
        
        print(f"\n✅ Parsing complete!")
        print(f"   📊 Statistics:")
        print(f"      event_city:    {cities:,}/{initial_count} ({cities/initial_count*100:.1f}%)")
        print(f"      event_state:   {states:,}/{initial_count} ({states/initial_count*100:.1f}%)")
        print(f"      event_country: {countries:,}/{initial_count} ({countries/initial_count*100:.1f}%)")
        
        # Примеры
        print(f"\n   📋 Sample parsed locations:")
        for idx in location_info.head(10).index:
            loc = location_info.at[idx, 'event_location']
            city = location_info.at[idx, 'event_city']
            state = location_info.at[idx, 'event_state']
            country = location_info.at[idx, 'event_country']
            
            state_str = f", {state}" if pd.notna(state) else ""
            country_str = f", {country}" if pd.notna(country) else ""
            
            print(f"      {loc[:40]:40} → {city}{state_str}{country_str}")
        
        logging.info(f"Parsed {initial_count:,} locations: {cities:,} cities, {states:,} states, {countries:,} countries")

print(f"\n✅ Cell 10.5 (Parse Location) complete!\n")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GEOCODING WITH CACHING - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

import json
from pathlib import Path

# Cache file для сохранения результатов геокодинга
GEOCODING_CACHE_FILE = 'geocoding_cache.json'

def load_geocoding_cache():
    """Загрузить кеш геокодинга из файла."""
    cache_path = Path(GEOCODING_CACHE_FILE)
    if cache_path.exists():
        try:
            with open(cache_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            logging.warning(f"Could not load geocoding cache: {e}")
    return {}

def save_geocoding_cache(cache):
    """Сохранить кеш геокодинга в файл."""
    try:
        with open(GEOCODING_CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)
    except Exception as e:
        logging.error(f"Could not save geocoding cache: {e}")

def get_coordinates_with_cache(location, gmaps_client=None, cache=None, use_cache=True):
    """
    Получить координаты локации с использованием кеша.
    
    Parameters:
    -----------
    location : str
        Название локации для геокодинга
    gmaps_client : googlemaps.Client, optional
        Google Maps API клиент
    cache : dict, optional
        Словарь с кешем координат
    use_cache : bool
        Использовать ли кеш (default: True)
        
    Returns:
    --------
    tuple or None
        (latitude, longitude) или None при ошибке
    """
    # Проверка кеша
    if use_cache and cache and location in cache:
        cached_result = cache[location]
        if cached_result is not None:
            return tuple(cached_result)  # Возвращаем как tuple
        return None
    
    # Если нет клиента, вернуть None
    if gmaps_client is None:
        return None
    
    try:
        # Задержка между запросами (rate limiting)
        time.sleep(REQUEST_DELAY)
        
        # Получение геоданных от Google Maps Geocoding API
        geocode_result = gmaps_client.geocode(location)
        
        # Если есть результат, возвращаем координаты
        if geocode_result and len(geocode_result) > 0:
            location_data = geocode_result[0]['geometry']['location']
            coords = (location_data['lat'], location_data['lng'])
            
            # Сохраняем в кеш
            if use_cache and cache is not None:
                cache[location] = list(coords)  # Сохраняем как list для JSON
            
            return coords
        else:
            # Сохраняем None в кеш, чтобы не запрашивать снова
            if use_cache and cache is not None:
                cache[location] = None
            return None
            
    except Exception as e:
        logging.error(f"Error fetching coordinates for '{location}': {type(e).__name__}: {e}")
        return None

print("\n🌍 Starting geocoding process...")

# Проверка наличия location_info
if 'location_info' not in globals():
    print(f"⚠️  Warning: location_info not found!")
    logging.warning(f"location_info not available")
elif not isinstance(location_info, pd.DataFrame):
    print(f"⚠️  Warning: location_info is not a DataFrame (type: {type(location_info).__name__})")
    logging.warning(f"location_info has wrong type: {type(location_info)}")
elif location_info.empty:
    print(f"⚠️  Warning: location_info DataFrame is empty!")
    logging.warning(f"location_info DataFrame is empty")
else:
    # Загрузка переменных среды
    load_dotenv()
    
    # Получение ключа API
    api_key = os.getenv('GOOGLE_MAPS_API_KEY')
    
    # Загрузка кеша
    geocoding_cache = load_geocoding_cache()
    cache_size = len(geocoding_cache)
    
    print(f"📦 Loaded geocoding cache: {cache_size:,} locations")
    
    # Проверка наличия API ключа
    if not api_key:
        print("⚠️  Warning: GOOGLE_MAPS_API_KEY not found in environment!")
        print("   Geocoding will use cache only (no new API requests)")
        gmaps_client = None
        use_api = False
    else:
        # Инициализация клиента Google Maps
        try:
            gmaps_client = googlemaps.Client(key=api_key)
            use_api = True
            print("✅ Google Maps API client initialized")
        except Exception as e:
            logging.error(f"Failed to initialize Google Maps client: {e}")
            gmaps_client = None
            use_api = False
            print("⚠️  Warning: Could not initialize Google Maps client. Using cache only.")
    
    # Проверка, нужны ли новые запросы
    locations_to_geocode = location_info['event_location'].tolist()
    locations_in_cache = set(geocoding_cache.keys())
    new_locations = [loc for loc in locations_to_geocode if loc not in locations_in_cache]
    
    print(f"\n📊 Geocoding statistics:")
    print(f"   Total locations: {len(locations_to_geocode):,}")
    print(f"   In cache: {len(locations_to_geocode) - len(new_locations):,}")
    print(f"   New locations: {len(new_locations):,}")
    
    if not use_api and new_locations:
        print(f"\n⚠️  Warning: {len(new_locations):,} locations not in cache, but API is not available!")
        print("   Coordinates will be None for these locations")
    
    # Применяем функцию к каждой локации с progress bar
    print(f"\n🔄 Geocoding locations...")
    
    coordinates_list = []
    stats = {
        'total': len(locations_to_geocode),
        'from_cache': 0,
        'from_api': 0,
        'errors': 0,
        'none': 0
    }
    
    # Progress bar
    pbar = tqdm(locations_to_geocode, desc="Geocoding", unit="location")
    
    for location in pbar:
        # Проверка кеша
        if location in geocoding_cache:
            cached_coords = geocoding_cache[location]
            if cached_coords is not None:
                coordinates_list.append(tuple(cached_coords))
                stats['from_cache'] += 1
            else:
                coordinates_list.append(None)
                stats['none'] += 1
        else:
            # Геокодинг через API
            coords = get_coordinates_with_cache(
                location, 
                gmaps_client=gmaps_client,
                cache=geocoding_cache,
                use_cache=True
            )
            coordinates_list.append(coords)
            
            if coords is not None:
                stats['from_api'] += 1
            else:
                stats['none'] += 1
        
        # Обновляем progress bar
        pbar.set_postfix({
            'cache': stats['from_cache'],
            'API': stats['from_api'],
            'none': stats['none']
        })
    
    pbar.close()
    
    # Сохраняем обновленный кеш
    if use_api:
        save_geocoding_cache(geocoding_cache)
        print(f"💾 Cache updated and saved")
    
    # Добавляем координаты в DataFrame
    location_info['Coordinates'] = coordinates_list
    
    # Разделяем координаты на два столбца (с обработкой None значений)
    location_info['latitude'] = location_info['Coordinates'].apply(lambda x: x[0] if x is not None else None)
    location_info['longitude'] = location_info['Coordinates'].apply(lambda x: x[1] if x is not None else None)
    
    # Удаление временного столбца с координатами
    location_info.drop(columns=['Coordinates'], inplace=True)
    
    # Финальная статистика
    print(f"\n✅ Geocoding complete!")
    print(f"   📊 Statistics:")
    print(f"      Total locations: {stats['total']:,}")
    print(f"      From cache: {stats['from_cache']:,} ({stats['from_cache']/stats['total']*100:.1f}%)")
    print(f"      From API: {stats['from_api']:,} ({stats['from_api']/stats['total']*100:.1f}%)")
    print(f"      No coordinates: {stats['none']:,} ({stats['none']/stats['total']*100:.1f}%)")
    print(f"   📦 Cache size: {len(geocoding_cache):,} locations")
    
    # Показываем примеры локаций без координат
    locations_without_coords = location_info[
        location_info['latitude'].isna() | location_info['longitude'].isna()
    ]
    if len(locations_without_coords) > 0:
        print(f"\n   ⚠️  Locations without coordinates ({len(locations_without_coords):,}):")
        sample_missing = locations_without_coords['event_location'].head(10).tolist()
        for loc in sample_missing:
            print(f"      - {loc}")
        if len(locations_without_coords) > 10:
            print(f"      ... and {len(locations_without_coords) - 10} more")
    
    logging.info(f"Geocoding complete: {stats['from_cache']} from cache, {stats['from_api']} from API, {stats['none']} missing")

print(f"\n✅ Cell 16 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVENT NAME NORMALIZATION - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🔄 Normalizing event names...")

# Проверка наличия DataFrame
if 'dancers_results_info' not in globals():
    print(f"⚠️  Warning: dancers_results_info not found!")
    logging.warning(f"dancers_results_info not available")
elif not isinstance(dancers_results_info, pd.DataFrame):
    print(f"⚠️  Warning: dancers_results_info is not a DataFrame (type: {type(dancers_results_info).__name__})")
    logging.warning(f"dancers_results_info has wrong type: {type(dancers_results_info)}")
elif dancers_results_info.empty:
    print(f"⚠️  Warning: dancers_results_info DataFrame is empty!")
    logging.warning(f"dancers_results_info DataFrame is empty")
else:
    initial_count = len(dancers_results_info)
    
    # Mapping для нормализации названий событий
    event_name_mapping = {
        'Arizona Dance Classic (Cancelled due to Covid-19)': 'Arizona Dance Classic',
        'Asia WCS Open - 10th Anniversary': 'Asia WCS Open',
        'Asia West Coast Swing Open': 'Asia WCS Open',
        'Asian WCS Open Swingvitation': 'Asia WCS Open',
        'Austin Swing Dance Championships (ASDC)': 'Austin Swing Dance Championships',
        'Bavarian Open West Coast Swing Championships': 'Bavarian Open',
        'Bavarian Open WCS': 'Bavarian Open',
        'Best of the Best WCS': 'Best of the Best',
        'Big Apple Dance Festival/World Hustle Championships': 'Big Apple Dance Festival',
        'Bridgetown Swing Boogie': 'BridgeTown Swing',
        'C.A.S.H. Bash': 'C.A.S.H. Bash Weekend',
        'Canadian Swing Championships (CSC)': 'Canadian Swing Championships',
        "Capital Swing Presidents' Day Convention": 'Capital Swing Dance Convention',
        "Capital Swing Dancers' President's Day": 'Capital Swing Dance Convention',
        'Charlotte WestieFest': 'Charlotte Westie Fest',
        'Chicagoland': 'Chicagoland Dance Festival',
        'Chicagoland Country & Swing Dance Festival': 'Chicagoland Dance Festival',
        'Chicagoland Country and Swing Dance Festival': 'Chicagoland Dance Festival',
        'Citadel Swing (Cancelled due to Covid-19)': 'Citadel Swing',
        'City of Angeles Swing Event': 'City of Angels',
        'D-Town Swing': 'D-Townswing',
        'DC Swing Experience (DCSX)': 'DC Swing eXperience',
        'DC Swing eXperience (DCSX)': 'DC Swing eXperience',
        'DC Swing eXperience  (DCSX)': 'DC Swing eXperience',
        'Desert City Swing Dance Championships': 'Desert City Swing',
        'DOWCS': 'Dutch Open',
        'Dutch Open West Coast Swing': 'Dutch Open',
        'Dutch Open West Coast Swing 2024': 'Dutch Open',
        'Floor Play Swing Vacation': 'Floorplay New Years Swing Vacation',
        'German Open WCS Championships': 'German Open',
        'German Open Championships': 'German Open',
        'Halloween Swingthing': 'Halloween SwingThing',
        "Jack & Jill O'Rama": "J&J O'Rama",
        'Korean Open Swing Dance Championships': 'Korean Open WCS Championsips',
        'LoneStar Invitational': 'Lone Star Invitational',
        'MADjam (Mid Atlantic Dance Jam)': 'MADjam',
        'Mid-Atlantic Dance Jam': 'MADjam',
        'Mid Atlantic Dance Jam (MADjam)': 'MADjam',
        'Meet Me in St. Louis Swing Dance Championships': 'Meet Me In St Louis',
        'Meet Me in St Louis Swing Dance Championships': 'Meet Me In St Louis',
        'Monterey SwingFest': 'Monterey Swing Fest',
        'Monterey Swingfest (Cancelled due to Covid-19)': 'Monterey Swing Fest',
        'Monterey Swingfest': 'Monterey Swing Fest',
        'Montreal WCS Fest': 'Montreal Westie Fest',
        'Moscow Westie Dance Fest': 'Moscow Westie Fest',
        'Mountain Magic Dance Convention': 'Mountain Magic',
        'New Zealand Open Swing Dance Championships': 'New Zealand Open',
        'Palm Springs New Years Swing Dance Classic': 'Palm Springs New Year',
        'Palm Springs Summer Dance Camp Classic': 'Palm Springs Swing Dance Classic',
        'Palm Springs Summer Dance Classic': 'Palm Springs Swing Dance Classic',
        'Paradise Country Dance Festival': 'Paradise Dance Festival',
        'Paradise Country and Swing Dance Festival': 'Paradise Dance Festival',
        'Paradise Country & Swing Dance Festival': 'Paradise Dance Festival',
        # 'Paris Swing Classic': 'Paris Westie Fest',
        '4TH of July Convention': 'Phoenix 4th of July',
        'Rocky Mountain Five Dance (RM5)': 'Rocky Mountain 5 Dance',
        'Russian Open WCS Championship': 'Russian Open',
        'Russian Open WCS Championships': 'Russian Open',
        'Scandinavian Open WCS': 'Scandinavian Open',
        'Scandinavian Open WCS 2022': 'Scandinavian Open',
        'Scandinavian Open WCS "SNOW"': 'Scandinavian Open',
        'Sea Sun & Swing Camp': 'Sea Sun and Swing',
        'Sea To Sky - Seattle': 'Sea to Sky',
        'Sea to Sky Seattle WCS': 'Sea to Sky',
        'Show-Me Showdown': 'Show Me Showdown',
        'Simply Adelaide West Coast Swing 2023': 'Simply Adelaide West Coast Swing',
        'Simply Adelaide West Coast Swing 2022': 'Simply Adelaide West Coast Swing',
        'SOswing 2022': 'SOswing',
        'South Bay CW Dance Festival': 'South Bay Dance Fling',
        "Spotlight New Year's Celebration": 'Spotlight Dance Challenge',
        "Swing Over Orlando": 'Swing Over',
        "Swing&Snow": 'Swing & Snow',
        "SwingCouver 2020 - The 10th Episode": 'SwingCouver',
        "SwingDiego (The Superbowl of Swing)": 'SwingDiego',
        "Swingin' New England Dance Festival": "Swingin' New England",
        "Swingtacular: The Galactic Open 2023": 'Swingtacular',
        "Swingtacular: The Galactic Open 2022": 'Swingtacular',
        "Swingtacular: The Galactic Open": 'Swingtacular',
        "Swingtimate 2020": 'Swingtimate',
        "SwingTime Denver": 'SwingTime',
        "Swingtime in the Rockies": 'SwingTime',
        'The After Party "TAP"': 'The After Party',
        "The After Party - TAP": 'The After Party',
        "The After Party (TAP)": 'The After Party',
        "The Brazilian Open Championships (Cancelled due to Coronavirus)": 'The Brazilian Open',
        "The Brazilian Open Championships": 'The Brazilian Open',
        "Toronto Open Swing & Hustle Championships": 'Toronto Open',
        "Toronto Open Swing and Hustle Championships": 'Toronto Open',
        "U.K. & European WCS Championships": 'UK & European WCS Championships',
        "USA Grand Nationals Dance Championships": 'USA Grand Nationals',
        "USA Grand National Dance Championships": 'USA Grand Nationals',
        "USA Grand Nationals Dance Championship": 'USA Grand Nationals',
        "Winter White WCS": 'Winter White',
        "Winter White West Coast Swing": 'Winter White',
        "The New Zealand Open": 'New Zealand Open',
        "Seattle's Easter Swing": 'Easter Swing',
        "Swing Trilogy": 'Trilogy Swing',
        "Rock the Barn": 'Rock The Barn',
        "Texas Classic": 'The Texas Classic',
        "Westies on the Water": 'Westies on The Water',
        "WESTY NANTES": 'Westy Nantes',
        "Country Dance World Championships": 'Worlds UCWDC',
        "UCWDC Country Dance World Championships": 'Worlds UCWDC',
        "New Orleans Dance Mardi Gras": 'Dance Mardi Gras',
        "The Boston Tea Party": 'Boston Tea Party',
        "Floorplay New Years Swing Vacation": 'FloorPlay New Years Swing Vacation',
        "Swingover": 'Swing Over',
        "Boogie by the Bay": 'Boogie By The Bay',
        "Saint Petersburg WCS Nights": 'St.Petersburg WCS Nights',
        "Sweden Westie Gala": 'Westie Gala',
        "CASH Bash": 'C.A.S.H. Bash Weekend',
        "London SWINGvitational": 'London SwingVitational',
        "Simply Adelaide West Coast Swing 2024": 'Simply Adelaide West Coast Swing',
        "Swingsation 2024": 'Swingsation',
        "The Australian Classic West Coast Swing Dance Championships (Trial Event)": 'The Australian Classic WCS Dance Championships',
        "SOM - Swing of Music": 'Swing of Music',
        "Swingvester": 'SwingVester',
        "By-Town Open (BTO)": 'BTO Open',
        "Swing Fiction 2024": 'Swing Fiction',
        "West in Lyon": 'West In Lyon',
        "French Open West Coast Swing": 'French Open WCS',
        "Global Grand Prix - West Coast Swing Reunion": 'Global Grand Prix - West Coast Swing',
        "D-TOWNSWING": 'D-Townswing',
        "The German Open WCS Championships": 'German Open',
        "Dutch open West Coast swing": 'Dutch Open',
        "NeverlandSwing": 'Neverland Swing',
        "BALTIC SWING": 'Baltic Swing',
        "KING SWING": 'King Swing',
        "Warsaw Swing": 'Warsaw Halloween Swing',
        "Korean Open WCS Championsips": 'Korean Open WCS Championships',
        "Americano Dance camp": 'Americano Dance Camp',
        "Asia WCS Open XI": 'Asia WCS Open',
        "UK WCS Dance Championships": 'UK WCS Championships',
        'The After Party - "TAP"': 'The After Party',
        'Monterey Swing Fest 2024': 'Monterey Swing Fest',
        "New Year's Dance Camp": 'Palm Springs New Year',
        "Capital Swing Convention": 'Capital Swing Dance Convention',
        "5280 Swing Dance Championships": '5280 Westival',
        "Philly Swing Dance Classic": 'Philly Swing Classic',
        "Florida Dance Magic (Unconfirmed)": 'Florida Dance Magic',
        "FloorPlay New Years Swing Vacation": 'Floorplay Swing Vacation',
        "Swing Dance America (SDA)": 'Swing Dance America',
        "SWINGAPALOOZA": 'Swingapalooza',
        "Michigan Dance Classic": 'Michigan Classic',
        "Swingin' Into Spring 2025": "Swingin' Into Spring",
        "UCWDC Country Dance World Championship": "Worlds UCWDC",
        "Swing Fling 2024": "Swing Fling",
        "Austin Rocks 2024": "Austin Rocks",
        "Lonestar Invitational": "Lone Star Invitational",
        "Midnight Madness Swing": "Midnight Madness",
        "Sea to Sky Seattle": "Sea to Sky",
        "St. Petersburg WCS Nights": "St.Petersburg WCS Nights",
        'Midwest Westie Fest 2025': 'Midwest Westie Fest',
        'UK & European WCS Championships': 'UK WCS Championships',
        'German Open West Coast Swing': 'German Open',
        'Korea Westival 2025': 'Korea Westival',
        'Milan Modern Swing 2025': 'Milan Modern Swing',
        'Mooseland Swing 2025': 'Mooseland Swing',
    }
    
    # Статистика изменений
    changes_count = 0
    changes_by_name = {}
    
    # Применяем замены
    if 'event_name' in dancers_results_info.columns:
        for old_name, new_name in event_name_mapping.items():
            mask = dancers_results_info['event_name'] == old_name
            count = mask.sum()
            if count > 0:
                dancers_results_info.loc[mask, 'event_name'] = new_name
                changes_count += count
                changes_by_name[old_name] = {'new': new_name, 'count': count}
                logging.info(f"Event name normalization: '{old_name}' → '{new_name}': {count:,} records")
    
    # Финальная статистика
    print(f"\n✅ Event name normalization complete!")
    print(f"   📊 Total records: {initial_count:,}")
    print(f"   🔄 Total changes: {changes_count:,}")
    print(f"   📋 Unique mappings applied: {len(changes_by_name)}")
    
    if changes_by_name:
        print(f"\n   📋 Top 10 event name changes:")
        sorted_changes = sorted(changes_by_name.items(), key=lambda x: x[1]['count'], reverse=True)
        for old_name, info in sorted_changes[:10]:
            print(f"      '{old_name}' → '{info['new']}': {info['count']:,} records")
    
    logging.info(f"Event name normalization: {changes_count:,} total changes applied")

print(f"\n✅ Cell 17 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STATE NAME REPLACEMENT - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

def replace_state(state_abbr):
    """
    Заменяет аббревиатуру штата на полное название.
    
    Parameters:
    -----------
    state_abbr : str
        Аббревиатура штата (например, 'CA', 'NY', 'DC')
        
    Returns:
    --------
    str or np.nan
        Полное название штата или NaN если не найдено
    """
    if state_abbr and not isinstance(state_abbr, float):
        # Специальная обработка для округа Колумбия
        if state_abbr == 'DC':
            return 'District of Columbia'
        
        # Поиск штата через библиотеку us
        try:
            state = us.states.lookup(state_abbr)
            return state.name if state else np.nan
        except Exception as e:
            logging.warning(f"Error looking up state '{state_abbr}': {e}")
            return np.nan
    return np.nan

print("\n🔄 Replacing state abbreviations with full names...")

# Проверка наличия location_info и колонки event_state
if 'location_info' not in globals():
    print(f"⚠️  Warning: location_info not found!")
    logging.warning(f"location_info not available")
elif not isinstance(location_info, pd.DataFrame):
    print(f"⚠️  Warning: location_info is not a DataFrame (type: {type(location_info).__name__})")
    logging.warning(f"location_info has wrong type: {type(location_info)}")
elif location_info.empty:
    print(f"⚠️  Warning: location_info DataFrame is empty!")
    logging.warning(f"location_info DataFrame is empty")
else:
    if 'event_state' not in location_info.columns:
        print("⚠️  Warning: 'event_state' column not found!")
        print("   State replacement cannot be performed without 'event_state' column")
        logging.warning("event_state column not found")
    else:
        initial_count = len(location_info)
        initial_nan_count = location_info['event_state'].isna().sum()
        initial_non_nan = initial_count - initial_nan_count
        
        # Применяем функцию
        print(f"   Processing {initial_non_nan:,} state abbreviations...")
        location_info['event_state'] = location_info['event_state'].apply(replace_state)
        
        # Статистика
        final_nan_count = location_info['event_state'].isna().sum()
        replaced_count = initial_non_nan - (final_nan_count - initial_nan_count)
        new_nan = final_nan_count - initial_nan_count
        
        print(f"\n✅ State replacement complete!")
        print(f"   📊 Statistics:")
        print(f"      Total locations: {initial_count:,}")
        print(f"      States replaced: {replaced_count:,}")
        print(f"      States not found (new NaN): {max(0, new_nan):,}")
        print(f"      Total missing states (NaN): {final_nan_count:,} ({final_nan_count/initial_count*100:.1f}%)")
        
        # Показываем уникальные штаты
        unique_states = location_info['event_state'].dropna().unique()
        if len(unique_states) > 0:
            print(f"      Unique states found: {len(unique_states)}")
            print(f"\n   📋 Sample states (first 10):")
            for state in sorted(unique_states)[:10]:
                count = (location_info['event_state'] == state).sum()
                print(f"      {state}: {count:,} locations")
        
        logging.info(f"State replacement: {replaced_count:,} states replaced, {new_nan} new NaN")

print(f"\n✅ Cell 14 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FILL MISSING STATES FOR US CITIES - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🔄 Filling missing states for US cities...")

# Проверка наличия location_info
if 'location_info' not in globals():
    print(f"⚠️  Warning: location_info not found!")
    logging.warning(f"location_info not available")
elif not isinstance(location_info, pd.DataFrame):
    print(f"⚠️  Warning: location_info is not a DataFrame (type: {type(location_info).__name__})")
    logging.warning(f"location_info has wrong type: {type(location_info)}")
elif location_info.empty:
    print(f"⚠️  Warning: location_info DataFrame is empty!")
    logging.warning(f"location_info DataFrame is empty")
else:
    # Проверяем наличие необходимых колонок
    required_cols = ['event_country', 'event_state', 'event_city']
    missing_cols = [col for col in required_cols if col not in location_info.columns]
    
    if missing_cols:
        print(f"⚠️  Warning: Missing columns: {missing_cols}")
        logging.warning(f"Missing columns for state filling: {missing_cols}")
    else:
        # Находим записи где страна = США, но штат отсутствует
        us_mask = (location_info['event_country'] == 'United States') | \
                  (location_info['event_country'] == 'USA') | \
                  (location_info['event_country'] == 'US')
        missing_state_mask = location_info['event_state'].isna()
        target_mask = us_mask & missing_state_mask & location_info['event_city'].notna()
        
        initial_missing = target_mask.sum()
        
        if initial_missing == 0:
            print("✅ No missing states found for US cities")
        else:
            print(f"   Found {initial_missing:,} US cities with missing states")
            
            # Метод 1: Словарь известных US городов -> штатов
            city_to_state_mapping = {
                # Major cities
                'New York': 'New York',
                'Los Angeles': 'California',
                'Chicago': 'Illinois',
                'Houston': 'Texas',
                'Phoenix': 'Arizona',
                'Philadelphia': 'Pennsylvania',
                'San Antonio': 'Texas',
                'San Diego': 'California',
                'Dallas': 'Texas',
                'San Jose': 'California',
                'Austin': 'Texas',
                'Jacksonville': 'Florida',
                'San Francisco': 'California',
                'Columbus': 'Ohio',
                'Fort Worth': 'Texas',
                'Charlotte': 'North Carolina',
                'Seattle': 'Washington',
                'Denver': 'Colorado',
                'Boston': 'Massachusetts',
                'El Paso': 'Texas',
                'Detroit': 'Michigan',
                'Nashville': 'Tennessee',
                'Portland': 'Oregon',
                'Memphis': 'Tennessee',
                'Oklahoma City': 'Oklahoma',
                'Las Vegas': 'Nevada',
                'Louisville': 'Kentucky',
                'Baltimore': 'Maryland',
                'Milwaukee': 'Wisconsin',
                'Albuquerque': 'New Mexico',
                'Tucson': 'Arizona',
                'Fresno': 'California',
                'Sacramento': 'California',
                'Kansas City': 'Missouri',
                'Mesa': 'Arizona',
                'Atlanta': 'Georgia',
                'Omaha': 'Nebraska',
                'Colorado Springs': 'Colorado',
                'Raleigh': 'North Carolina',
                'Virginia Beach': 'Virginia',
                'Miami': 'Florida',
                'Oakland': 'California',
                'Minneapolis': 'Minnesota',
                'Tulsa': 'Oklahoma',
                'Cleveland': 'Ohio',
                'Wichita': 'Kansas',
                'Arlington': 'Texas',
                'New Orleans': 'Louisiana',
                'Tampa': 'Florida',
                'Honolulu': 'Hawaii',
                # Additional common event cities
                'Palm Springs': 'California',
                'Monterey': 'California',
                'San Bernardino': 'California',
                'Buena Park': 'California',
                'Irvine': 'California',
                'Newton': 'Massachusetts',
                'Costa Mesa': 'California',
                'Burbank': 'California',
                'Framingham': 'Massachusetts',
                'Cape Cod': 'Massachusetts',
                'Long Beach': 'California',
                'Reston': 'Virginia',
                'Herndon': 'Virginia',
            }
            
            filled_from_dict = 0
            
            # Применяем mapping
            for idx in location_info[target_mask].index:
                city = location_info.loc[idx, 'event_city']
                if pd.notna(city) and city in city_to_state_mapping:
                    location_info.loc[idx, 'event_state'] = city_to_state_mapping[city]
                    filled_from_dict += 1
                    logging.debug(f"Filled state for {city}: {city_to_state_mapping[city]}")
            
            # Метод 2: Обратный геокодинг через координаты (если есть)
            filled_from_geocoding = 0
            
            if 'latitude' in location_info.columns and 'longitude' in location_info.columns:
                # Обновляем маску после заполнения из словаря
                still_missing_mask = us_mask & location_info['event_state'].isna() & location_info['event_city'].notna()
                
                missing_with_coords = still_missing_mask & \
                                     location_info['latitude'].notna() & \
                                     location_info['longitude'].notna()
                
                if missing_with_coords.sum() > 0:
                    print(f"   Attempting reverse geocoding for {missing_with_coords.sum():,} locations with coordinates...")
                    
                    # Используем Nominatim (бесплатный) для обратного геокодинга
                    try:
                        from geopy.geocoders import Nominatim
                        geolocator = Nominatim(user_agent="wsdc_parser")
                        
                        # Обрабатываем небольшими батчами чтобы не превысить rate limits
                        coords_to_process = location_info[missing_with_coords][['latitude', 'longitude', 'event_city']].head(100)
                        
                        for idx, row in coords_to_process.iterrows():
                            try:
                                lat, lon = row['latitude'], row['longitude']
                                location = geolocator.reverse(f"{lat}, {lon}", exactly_one=True, timeout=5)
                                
                                if location:
                                    address = location.raw.get('address', {})
                                    state = address.get('state') or address.get('region')
                                    
                                    if state:
                                        # Преобразуем в полное название если нужно
                                        try:
                                            state_obj = us.states.lookup(state)
                                            if state_obj:
                                                location_info.loc[idx, 'event_state'] = state_obj.name
                                                filled_from_geocoding += 1
                                                logging.debug(f"Geocoded state for {row['event_city']}: {state_obj.name}")
                                        except:
                                            location_info.loc[idx, 'event_state'] = state
                                            filled_from_geocoding += 1
                                
                                time.sleep(1)  # Rate limiting
                            except Exception as e:
                                logging.warning(f"Error in reverse geocoding for index {idx}: {e}")
                                continue
                                
                    except Exception as e:
                        print(f"   ⚠️  Reverse geocoding not available: {e}")
                        logging.warning(f"Reverse geocoding failed: {e}")
            else:
                print("   ⚠️  Coordinates not available for reverse geocoding")
            
            # Финальная статистика
            final_missing = (us_mask & location_info['event_state'].isna() & location_info['event_city'].notna()).sum()
            total_filled = filled_from_dict + filled_from_geocoding
            
            print(f"\n✅ State filling complete!")
            print(f"   📊 Statistics:")
            print(f"      Initial missing states: {initial_missing:,}")
            print(f"      Filled from dictionary: {filled_from_dict:,}")
            print(f"      Filled from geocoding: {filled_from_geocoding:,}")
            print(f"      Total filled: {total_filled:,}")
            print(f"      Still missing: {final_missing:,}")
            
            if total_filled > 0:
                print(f"   ✅ Successfully filled {total_filled/initial_missing*100:.1f}% of missing states")
            
            logging.info(f"State filling: {filled_from_dict} from dict, {filled_from_geocoding} from geocoding, {final_missing} still missing")

print(f"\n✅ Cell 15 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DISAMBIGUATE DUPLICATE CITY NAMES - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════
#
# 🎯 НАЗНАЧЕНИЕ: Добавление уточнения страны к городам с одинаковыми
#                названиями в разных странах
#
# 📋 ПРИМЕРЫ:
#    • St. Petersburg (Russia) vs St. Petersburg (USA)
#    • Vancouver (Canada) vs Vancouver (USA)
#
# ⚙️  ЛОГИКА:
#    1. Находит города, которые существуют в нескольких странах
#    2. Добавляет (Страна) к названию города для различия
# ═══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import logging

print("\n🔄 Disambiguating duplicate city names...")

if 'location_info' not in globals():
    print("⚠️  Warning: location_info not found!")
    logging.warning("location_info not available")
elif not isinstance(location_info, pd.DataFrame):
    print(f"⚠️  Warning: location_info is not a DataFrame (type: {type(location_info).__name__})!")
    logging.warning(f"location_info has wrong type: {type(location_info)}")
elif location_info.empty:
    print("⚠️  Warning: location_info DataFrame is empty!")
    logging.warning("location_info DataFrame is empty")
else:
    if 'event_city' not in location_info.columns or 'event_country' not in location_info.columns:
        print("⚠️  Warning: Required columns not found!")
        logging.warning("event_city or event_country columns not found")
    else:
        initial_count = len(location_info)
        print(f"   Processing {initial_count:,} locations...")
        
        # ═══════════════════════════════════════════════════════════════
        # Автоматическое обнаружение городов-дубликатов
        # ═══════════════════════════════════════════════════════════════
        
        # Находим города, которые есть в нескольких странах
        city_country_counts = location_info.groupby('event_city')['event_country'].nunique()
        duplicate_cities = city_country_counts[city_country_counts > 1].index.tolist()
        
        # Убираем NaN города
        duplicate_cities = [city for city in duplicate_cities if pd.notna(city)]
        
        if duplicate_cities:
            print(f"\n   🔍 Found {len(duplicate_cities)} cities with duplicate names across countries:")
            for city in sorted(duplicate_cities)[:10]:  # Показываем первые 10
                countries = location_info[location_info['event_city'] == city]['event_country'].unique()
                countries_str = ', '.join([c for c in countries if pd.notna(c)])
                print(f"      • {city}: {countries_str}")
            
            if len(duplicate_cities) > 10:
                print(f"      ... and {len(duplicate_cities) - 10} more")
        
        # ═══════════════════════════════════════════════════════════════
        # Добавляем уточнение страны
        # ═══════════════════════════════════════════════════════════════
        
        changes_count = 0
        
        for city in duplicate_cities:
            # Получаем все страны для этого города
            city_mask = location_info['event_city'] == city
            countries = location_info.loc[city_mask, 'event_country'].unique()
            
            # Для каждой страны добавляем уточнение
            for country in countries:
                if pd.notna(country):
                    mask = (location_info['event_city'] == city) & (location_info['event_country'] == country)
                    count_before = mask.sum()
                    
                    if count_before > 0:
                        # Сокращаем название страны для краткости
                        country_short = country
                        if country == 'United States':
                            country_short = 'USA'
                        elif country == 'United Kingdom':
                            country_short = 'UK'
                        
                        # Добавляем уточнение
                        location_info.loc[mask, 'event_city'] = f"{city} ({country_short})"
                        changes_count += count_before
        
        # ═══════════════════════════════════════════════════════════════
        # Статистика
        # ═══════════════════════════════════════════════════════════════
        
        print(f"\n✅ City disambiguation complete!")
        print(f"   📊 Statistics:")
        print(f"      Cities with duplicates: {len(duplicate_cities)}")
        print(f"      Records updated: {changes_count:,}/{initial_count:,} ({changes_count/initial_count*100:.1f}%)")
        
        # Примеры обновленных городов
        if changes_count > 0:
            print(f"\n   📋 Sample disambiguated cities:")
            for city in sorted(duplicate_cities)[:5]:
                updated_cities = location_info[
                    location_info['event_city'].str.contains(city, na=False, regex=False)
                ]['event_city'].unique()
                for updated_city in updated_cities:
                    print(f"      {updated_city}")
        
        logging.info(f"Disambiguated {len(duplicate_cities)} city names: {changes_count:,} records updated")

print(f"\n✅ Cell 15.5 (City Disambiguation) complete!\n")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EXTRACT CITY FROM LOCATION - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

def extract_city(location):
    """
    Извлекает название города из строки локации.
    
    Parameters:
    -----------
    location : str
        Строка локации в формате "City, State, Country" или подобном
        
    Returns:
    --------
    str or np.nan
        Название города или NaN если локация пустая/невалидная
    """
    if not location or pd.isna(location):
        return np.nan
    
    try:
        # Разделяем по запятой и пробелу
        parts = str(location).split(', ')
        
        # Берем первую часть как город
        city = parts[0].strip()
        
        # Возвращаем город если он не пустой
        return city if city else np.nan
    except Exception as e:
        logging.warning(f"Error extracting city from '{location}': {e}")
        return np.nan

print("\n🔄 Extracting cities from locations...")

# Проверка наличия location_info и колонки event_location
if 'location_info' not in globals():
    print(f"⚠️  Warning: location_info not found!")
    logging.warning(f"location_info not available")
elif not isinstance(location_info, pd.DataFrame):
    print(f"⚠️  Warning: location_info is not a DataFrame (type: {type(location_info).__name__})")
    logging.warning(f"location_info has wrong type: {type(location_info)}")
elif location_info.empty:
    print(f"⚠️  Warning: location_info DataFrame is empty!")
    logging.warning(f"location_info DataFrame is empty")
else:
    if 'event_location' not in location_info.columns:
        print("⚠️  Warning: 'event_location' column not found!")
        print("   City extraction cannot be performed without 'event_location' column")
        logging.warning("event_location column not found")
    else:
        initial_count = len(location_info)
        
        # Применяем функцию
        print(f"   Processing {initial_count:,} locations...")
        location_info['event_city'] = location_info['event_location'].apply(extract_city)
        
        # Статистика
        cities_extracted = location_info['event_city'].notna().sum()
        nan_count = location_info['event_city'].isna().sum()
        unique_cities = location_info['event_city'].nunique()
        
        print(f"\n✅ City extraction complete!")
        print(f"   📊 Statistics:")
        print(f"      Total locations: {initial_count:,}")
        print(f"      Cities extracted: {cities_extracted:,} ({cities_extracted/initial_count*100:.1f}%)")
        print(f"      Missing cities (NaN): {nan_count:,} ({nan_count/initial_count*100:.1f}%)")
        print(f"      Unique cities: {unique_cities:,}")
        
        # Показываем примеры городов
        if cities_extracted > 0:
            print(f"\n   📋 Sample cities (first 15):")
            sample_cities = location_info['event_city'].dropna().value_counts().head(15)
            for city, count in sample_cities.items():
                print(f"      {city}: {count:,} locations")
        
        logging.info(f"City extraction: {cities_extracted:,} cities extracted, {nan_count:,} missing")

print(f"\n✅ Cell 17 complete!")


In [ ]:
location_info['location_id'] = location_info.reset_index().index + 1

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EXTRACT MONTH AND YEAR FROM DATE - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

import numpy as np

def extract_month(date_string):
    """
    Извлекает месяц из строки даты.
    
    Parameters:
    -----------
    date_string : str
        Строка даты в формате "Month Year" (например, "January 2020")
        
    Returns:
    --------
    str or np.nan
        Название месяца или NaN если дата пустая/невалидная
    """
    if not date_string or pd.isna(date_string):
        return np.nan
    
    try:
        parts = str(date_string).strip().split()
        if len(parts) >= 1:
            return parts[0].strip()
        return np.nan
    except Exception as e:
        logging.warning(f"Error extracting month from '{date_string}': {e}")
        return np.nan

def extract_year(date_string):
    """
    Извлекает год из строки даты.
    
    Parameters:
    -----------
    date_string : str
        Строка даты в формате "Month Year" (например, "January 2020")
        
    Returns:
    --------
    str or np.nan
        Год или NaN если дата пустая/невалидная
    """
    if not date_string or pd.isna(date_string):
        return np.nan
    
    try:
        parts = str(date_string).strip().split()
        if len(parts) >= 2:
            return parts[1].strip()
        return np.nan
    except Exception as e:
        logging.warning(f"Error extracting year from '{date_string}': {e}")
        return np.nan

print("\n🔄 Extracting month and year from event dates...")

# Проверка наличия dancers_results_info
if 'dancers_results_info' not in globals():
    print(f"⚠️  Warning: dancers_results_info not found!")
    logging.warning(f"dancers_results_info not available")
elif not isinstance(dancers_results_info, pd.DataFrame):
    print(f"⚠️  Warning: dancers_results_info is not a DataFrame (type: {type(dancers_results_info).__name__})")
    logging.warning(f"dancers_results_info has wrong type: {type(dancers_results_info)}")
elif dancers_results_info.empty:
    print(f"⚠️  Warning: dancers_results_info DataFrame is empty!")
    logging.warning(f"dancers_results_info DataFrame is empty")
else:
    if 'event_date' not in dancers_results_info.columns:
        print("⚠️  Warning: 'event_date' column not found!")
        logging.warning("event_date column not found")
    else:
        initial_count = len(dancers_results_info)
        
        print(f"   Processing {initial_count:,} records...")
        
        # Применяем функции
        dancers_results_info['event_month'] = dancers_results_info['event_date'].apply(extract_month)
        dancers_results_info['event_year'] = dancers_results_info['event_date'].apply(extract_year)
        
        # Статистика
        months_extracted = dancers_results_info['event_month'].notna().sum()
        years_extracted = dancers_results_info['event_year'].notna().sum()
        
        month_missing = dancers_results_info['event_month'].isna().sum()
        year_missing = dancers_results_info['event_year'].isna().sum()
        
        print(f"\n✅ Date extraction complete!")
        print(f"   📊 Statistics:")
        print(f"      Total records: {initial_count:,}")
        print(f"      Months extracted: {months_extracted:,} ({months_extracted/initial_count*100:.1f}%)")
        print(f"      Years extracted: {years_extracted:,} ({years_extracted/initial_count*100:.1f}%)")
        
        if month_missing > 0 or year_missing > 0:
            print(f"      Missing months: {month_missing:,} ({month_missing/initial_count*100:.1f}%)")
            print(f"      Missing years: {year_missing:,} ({year_missing/initial_count*100:.1f}%)")
        
        # Показываем уникальные значения
        unique_months = dancers_results_info['event_month'].dropna().unique()
        unique_years = dancers_results_info['event_year'].dropna().unique()
        
        if len(unique_months) > 0:
            print(f"\n   📅 Unique months found: {len(unique_months)}")
            print(f"      {sorted(unique_months)[:12]}")
        
        if len(unique_years) > 0:
            print(f"\n   📅 Unique years found: {len(unique_years)}")
            years_list = sorted(unique_years)
            print(f"      Range: {years_list[0]} - {years_list[-1]}")
        
        logging.info(f"Date extraction: {months_extracted:,} months, {years_extracted:,} years extracted")

print(f"\n✅ Cell 20 complete!")


In [ ]:
dancers_results_info = dancers_results_info.merge(location_info[['event_location', 'location_id']], on='event_location', how='left')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CONVERT MONTH/YEAR TO DATETIME AND PROCESS EVENT POINTS - OPTIMIZED
# ═══════════════════════════════════════════════════════════════════

print("\n🔄 Converting months to numbers and creating datetime...")

# Проверка наличия dancers_results_info
if 'dancers_results_info' not in globals():
    print(f"⚠️  Warning: dancers_results_info not found!")
    logging.warning(f"dancers_results_info not available")
elif not isinstance(dancers_results_info, pd.DataFrame):
    print(f"⚠️  Warning: dancers_results_info is not a DataFrame (type: {type(dancers_results_info).__name__})")
    logging.warning(f"dancers_results_info has wrong type: {type(dancers_results_info)}")
elif dancers_results_info.empty:
    print(f"⚠️  Warning: dancers_results_info DataFrame is empty!")
    logging.warning(f"dancers_results_info DataFrame is empty")
else:
    # Проверка наличия необходимых колонок
    required_cols = ['event_month', 'event_year', 'event_points']
    missing_cols = [col for col in required_cols if col not in dancers_results_info.columns]
    
    if missing_cols:
        print(f"⚠️  Warning: Missing columns: {missing_cols}")
        logging.warning(f"Missing columns for datetime conversion: {missing_cols}")
    else:
        initial_count = len(dancers_results_info)
        print(f"   Processing {initial_count:,} records...")
        
        # Словарь соответствия месяцев
        month_dict = {
            'January': 1, 'February': 2, 'March': 3, 'April': 4, 
            'May': 5, 'June': 6, 'July': 7, 'August': 8, 
            'September': 9, 'October': 10, 'November': 11, 'December': 12
        }
        
        # 1. Преобразуем месяц в число
        print("\n   📅 Converting month names to numbers...")
        dancers_results_info['event_month'] = dancers_results_info['event_month'].map(month_dict)
        
        months_converted = dancers_results_info['event_month'].notna().sum()
        months_missing = dancers_results_info['event_month'].isna().sum()
        
        print(f"      Converted: {months_converted:,} ({months_converted/initial_count*100:.1f}%)")
        if months_missing > 0:
            print(f"      ⚠️  Missing: {months_missing:,} ({months_missing/initial_count*100:.1f}%)")
        
        # 2. Обработка event_year
        print("\n   📅 Processing event_year...")
        year_before = dancers_results_info['event_year'].notna().sum()
        
        dancers_results_info['event_year'] = pd.to_numeric(
            dancers_results_info['event_year'], 
            errors='coerce'
        )
        
        year_after_numeric = dancers_results_info['event_year'].notna().sum()
        year_invalid = year_before - year_after_numeric
        
        if year_invalid > 0:
            print(f"      ⚠️  Invalid years converted to NaN: {year_invalid}")
            logging.warning(f"Found {year_invalid} invalid year values")
        
        # Заполняем NaN значениями по умолчанию
        year_nan_before = dancers_results_info['event_year'].isna().sum()
        month_nan_before = dancers_results_info['event_month'].isna().sum()
        
        dancers_results_info['event_year'] = dancers_results_info['event_year'].fillna(1900).astype(int)
        dancers_results_info['event_month'] = dancers_results_info['event_month'].fillna(1).astype(int)
        
        if year_nan_before > 0 or month_nan_before > 0:
            print(f"\n   ⚠️  Filled missing values with defaults:")
            if year_nan_before > 0:
                print(f"      Year (1900): {year_nan_before:,}")
            if month_nan_before > 0:
                print(f"      Month (January): {month_nan_before:,}")
        
        # 3. Создаём event_year_and_month
        print("\n   📅 Creating event_year_and_month datetime...")
        dancers_results_info['event_year_and_month'] = pd.to_datetime(
            dancers_results_info['event_year'].astype(str) + '-' + 
            dancers_results_info['event_month'].astype(str),
            format='%Y-%m', 
            errors='coerce'
        )
        
        datetime_created = dancers_results_info['event_year_and_month'].notna().sum()
        datetime_failed = dancers_results_info['event_year_and_month'].isna().sum()
        
        print(f"      Created: {datetime_created:,} ({datetime_created/initial_count*100:.1f}%)")
        if datetime_failed > 0:
            print(f"      ⚠️  Failed: {datetime_failed:,} ({datetime_failed/initial_count*100:.1f}%)")
        
        # Показываем диапазон дат
        if datetime_created > 0:
            min_date = dancers_results_info['event_year_and_month'].min()
            max_date = dancers_results_info['event_year_and_month'].max()
            print(f"      Range: {min_date.strftime('%Y-%m')} to {max_date.strftime('%Y-%m')}")
            
            # Проверка на подозрительные даты (1900)
            suspicious_dates = dancers_results_info[
                dancers_results_info['event_year'] == 1900
            ]
            if len(suspicious_dates) > 0:
                print(f"      ⚠️  Records with year 1900: {len(suspicious_dates):,}")
        
        # 4. Обработка event_points
        print("\n   🎯 Processing event_points...")
        points_before = dancers_results_info['event_points'].notna().sum()
        
        dancers_results_info['event_points'] = pd.to_numeric(
            dancers_results_info['event_points'], 
            errors='coerce'
        ).fillna(0).astype(int)
        
        points_after = (dancers_results_info['event_points'] != 0).sum()
        points_zero = (dancers_results_info['event_points'] == 0).sum()
        
        print(f"      Non-zero points: {points_after:,} ({points_after/initial_count*100:.1f}%)")
        if points_zero > 0:
            print(f"      Zero points: {points_zero:,} ({points_zero/initial_count*100:.1f}%)")
        
        # Статистика по очкам
        if points_after > 0:
            avg_points = dancers_results_info[dancers_results_info['event_points'] > 0]['event_points'].mean()
            max_points = dancers_results_info['event_points'].max()
            print(f"      Average (non-zero): {avg_points:.1f}")
            print(f"      Maximum: {max_points}")
        
        # Итоговая статистика
        print(f"\n✅ Datetime and points conversion complete!")
        print(f"   📊 Summary:")
        print(f"      Total records: {initial_count:,}")
        print(f"      Valid datetimes: {datetime_created:,} ({datetime_created/initial_count*100:.1f}%)")
        print(f"      Valid points: {points_after:,} ({points_after/initial_count*100:.1f}%)")
        
        # Показываем примеры
        print(f"\n   📋 Sample data:")
        sample_cols = ['event_year', 'event_month', 'event_year_and_month', 'event_points']
        print(dancers_results_info[sample_cols].head().to_string(index=False))
        
        logging.info(f"Datetime conversion: {datetime_created:,} records, range {min_date} to {max_date}")
        logging.info(f"Points conversion: {points_after:,} non-zero points, avg {avg_points:.1f}")

print(f"\n✅ Cell 23 complete!")


In [ ]:
dancers_results_info[dancers_results_info['event_name'] == "Swingin' Into Spring"]
event_years = dancers_results_info[dancers_results_info['event_name'] == "Swingin' Into Spring"]['event_year'].unique()
print(sorted(event_years))  # Выведет годы в отсортированном порядке

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIX BROKEN DATES - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🔧 Fixing broken dates in data...")

# Проверка наличия dancers_results_info
if 'dancers_results_info' not in globals():
    print(f"⚠️  Warning: dancers_results_info not found!")
    logging.warning(f"dancers_results_info not available")
elif not isinstance(dancers_results_info, pd.DataFrame):
    print(f"⚠️  Warning: dancers_results_info is not a DataFrame (type: {type(dancers_results_info).__name__})")
    logging.warning(f"dancers_results_info has wrong type: {type(dancers_results_info)}")
elif dancers_results_info.empty:
    print(f"⚠️  Warning: dancers_results_info DataFrame is empty!")
    logging.warning(f"dancers_results_info DataFrame is empty")
else:
    # Проверка наличия необходимых колонок
    required_cols = ['event_name', 'event_date', 'event_year', 'event_month', 'event_year_and_month']
    missing_cols = [col for col in required_cols if col not in dancers_results_info.columns]
    
    if missing_cols:
        print(f"⚠️  Warning: Missing columns: {missing_cols}")
        logging.warning(f"Missing columns for date fixing: {missing_cols}")
    else:
        try:
            # Список известных битых дат для исправления
            date_fixes = [
                {
                    'event_name': 'Summer Hummer',
                    'broken_date': 'November -0001',
                    'correct_date': 'August 2025',
                    'year': 2025,
                    'month': 8
                }
                # Можно добавить другие битые даты здесь
            ]
            
            total_fixed = 0
            
            for fix in date_fixes:
                # Находим записи с битой датой
                mask = (
                    (dancers_results_info['event_name'] == fix['event_name']) & 
                    (dancers_results_info['event_date'] == fix['broken_date'])
                )
                
                affected_count = mask.sum()
                
                if affected_count > 0:
                    print(f"\n   🔧 Fixing: {fix['event_name']}")
                    print(f"      Broken date: '{fix['broken_date']}'")
                    print(f"      Correct date: '{fix['correct_date']}'")
                    print(f"      Affected records: {affected_count:,}")
                    
                    # Применяем исправления
                    dancers_results_info.loc[mask, 'event_date'] = fix['correct_date']
                    dancers_results_info.loc[mask, 'event_year'] = fix['year']
                    dancers_results_info.loc[mask, 'event_month'] = fix['month']
                    dancers_results_info.loc[mask, 'event_year_and_month'] = pd.to_datetime(
                        f"{fix['year']}-{fix['month']:02d}-01"
                    )
                    
                    total_fixed += affected_count
                    
                    # Показываем пример исправленных данных
                    sample = dancers_results_info.loc[
                        mask,
                        ['event_name', 'event_date', 'event_year', 'event_month', 'event_year_and_month']
                    ].drop_duplicates().head(3)
                    
                    print(f"\n      ✅ Fixed data sample:")
                    for col in sample.columns:
                        print(f"         {col}: {sample[col].iloc[0]}")
                    
                    logging.info(
                        f"Fixed {affected_count} records for {fix['event_name']}: "
                        f"{fix['broken_date']} -> {fix['correct_date']}"
                    )
                else:
                    print(f"\n   ℹ️  {fix['event_name']}: No broken dates found (already fixed or not present)")
            
            # Итоговая статистика
            if total_fixed > 0:
                print(f"\n✅ Date fixing complete!")
                print(f"   📊 Total records fixed: {total_fixed:,}")
            else:
                print(f"\n✅ No broken dates found in the data.")
            
        except Exception as e:
            print(f"❌ Error fixing dates: {type(e).__name__}: {e}")
            logging.error(f"Error in date fixing: {type(e).__name__}: {e}")

print(f"\n✅ Cell 21 complete!")


In [ ]:
# Фильтруем строки, где event_year_and_month является NaT (null)
null_rows = dancers_results_info[dancers_results_info['event_year_and_month'].isna()]
null_rows

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# REORDER LOCATION_INFO COLUMNS - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🔄 Reordering location_info columns...")

# Проверка наличия location_info
if 'location_info' not in globals():
    print(f"⚠️  Warning: location_info not found!")
    logging.warning(f"location_info not available")
elif not isinstance(location_info, pd.DataFrame):
    print(f"⚠️  Warning: location_info is not a DataFrame (type: {type(location_info).__name__})")
    logging.warning(f"location_info has wrong type: {type(location_info)}")
elif location_info.empty:
    print(f"⚠️  Warning: location_info DataFrame is empty!")
    logging.warning(f"location_info DataFrame is empty")
else:
    try:
        # Желаемый порядок колонок
        desired_columns = [
            'location_id', 
            'event_city', 
            'event_state',
            'event_country',
            'latitude',
            'longitude',
            'event_location'
        ]
        
        # Проверяем наличие всех колонок
        current_columns = set(location_info.columns)
        desired_set = set(desired_columns)
        
        missing = desired_set - current_columns
        extra = current_columns - desired_set
        
        if missing:
            print(f"⚠️  Warning: Missing columns: {missing}")
            # Используем только существующие колонки
            desired_columns = [col for col in desired_columns if col in current_columns]
        
        if extra:
            print(f"   ℹ️  Extra columns will be dropped: {extra}")
            logging.info(f"Dropping extra columns from location_info: {extra}")
        
        # Переупорядочиваем
        initial_shape = location_info.shape
        location_info = location_info.reindex(columns=desired_columns)
        final_shape = location_info.shape
        
        print(f"\n✅ Columns reordered!")
        print(f"   📊 Shape: {initial_shape} -> {final_shape}")
        print(f"   📋 Column order: {list(location_info.columns)}")
        
        if extra:
            print(f"   🗑️  Dropped {len(extra)} extra column(s)")
        
        logging.info(f"location_info columns reordered: {initial_shape[1]} -> {final_shape[1]} columns")
        
    except Exception as e:
        print(f"❌ Error reordering columns: {type(e).__name__}: {e}")
        logging.error(f"Error in column reordering: {type(e).__name__}: {e}")

print(f"\n✅ Cell 24 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DROP UNNECESSARY COLUMNS - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🗑️  Dropping unnecessary columns from dancers_results_info...")

# Проверка наличия dancers_results_info
if 'dancers_results_info' not in globals():
    print(f"⚠️  Warning: dancers_results_info not found!")
    logging.warning(f"dancers_results_info not available")
elif not isinstance(dancers_results_info, pd.DataFrame):
    print(f"⚠️  Warning: dancers_results_info is not a DataFrame (type: {type(dancers_results_info).__name__})")
    logging.warning(f"dancers_results_info has wrong type: {type(dancers_results_info)}")
elif dancers_results_info.empty:
    print(f"⚠️  Warning: dancers_results_info DataFrame is empty!")
    logging.warning(f"dancers_results_info DataFrame is empty")
else:
    try:
        # Колонки для удаления
        columns_to_drop = ['event_location', 'event_date']
        
        # Проверяем какие из них существуют
        existing_to_drop = [col for col in columns_to_drop if col in dancers_results_info.columns]
        not_found = [col for col in columns_to_drop if col not in dancers_results_info.columns]
        
        if not_found:
            print(f"   ℹ️  Columns not found (already dropped?): {not_found}")
        
        if existing_to_drop:
            initial_shape = dancers_results_info.shape
            dancers_results_info = dancers_results_info.drop(columns=existing_to_drop)
            final_shape = dancers_results_info.shape
            
            print(f"\n✅ Columns dropped successfully!")
            print(f"   📊 Shape: {initial_shape} -> {final_shape}")
            print(f"   🗑️  Dropped: {existing_to_drop}")
            print(f"   📋 Remaining columns: {len(dancers_results_info.columns)}")
            
            logging.info(f"Dropped columns from dancers_results_info: {existing_to_drop}")
        else:
            print(f"\n   ℹ️  No columns to drop (all already removed)")
        
    except Exception as e:
        print(f"❌ Error dropping columns: {type(e).__name__}: {e}")
        logging.error(f"Error dropping columns: {type(e).__name__}: {e}")

print(f"\n✅ Cell 25 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# REMOVE DUPLICATE ENTRIES (DANCERS WITH 2 ROLES IN SAME EVENT)
# ═══════════════════════════════════════════════════════════════════

print("\n🔍 Removing duplicate entries (dancers with 2 roles at same event)...")

# Проверка наличия dancers_results_info
if 'dancers_results_info' not in globals():
    print(f"⚠️  Warning: dancers_results_info not found!")
    logging.warning(f"dancers_results_info not available")
elif not isinstance(dancers_results_info, pd.DataFrame):
    print(f"⚠️  Warning: dancers_results_info is not a DataFrame (type: {type(dancers_results_info).__name__})")
    logging.warning(f"dancers_results_info has wrong type: {type(dancers_results_info)}")
elif dancers_results_info.empty:
    print(f"⚠️  Warning: dancers_results_info DataFrame is empty!")
    logging.warning(f"dancers_results_info DataFrame is empty")
else:
    try:
        initial_count = len(dancers_results_info)
        initial_shape = dancers_results_info.shape
        
        # Определяем колонки для проверки дубликатов (все кроме роли)
        columns_to_check = [col for col in dancers_results_info.columns if col != 'dancer_role']
        
        print(f"   Initial records: {initial_count:,}")
        print(f"   Checking duplicates by: {len(columns_to_check)} columns (excluding 'dancer_role')")
        
        # Подсчёт дубликатов до удаления
        duplicates_mask = dancers_results_info.duplicated(subset=columns_to_check, keep='first')
        duplicates_count = duplicates_mask.sum()
        
        if duplicates_count > 0:
            print(f"\n   🔍 Found {duplicates_count:,} duplicate entries")
            
            # Удаляем дубликаты
            dancers_results_info = dancers_results_info.drop_duplicates(subset=columns_to_check, keep='first')
            
            final_count = len(dancers_results_info)
            final_shape = dancers_results_info.shape
            
            print(f"\n✅ Duplicates removed!")
            print(f"   📊 Shape: {initial_shape} -> {final_shape}")
            print(f"   🗑️  Removed: {duplicates_count:,} records")
            print(f"   ✅ Remaining: {final_count:,} records")
            
            logging.info(f"Removed {duplicates_count:,} duplicate entries from dancers_results_info")
        else:
            print(f"\n   ℹ️  No duplicates found!")
            logging.info("No duplicates found in dancers_results_info")
        
    except Exception as e:
        print(f"❌ Error removing duplicates: {type(e).__name__}: {e}")
        logging.error(f"Error in deduplication: {type(e).__name__}: {e}")

print(f"\n✅ Cell 26 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DETECT CHANGES IN DANCER_ROLE_INFO - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🔍 Detecting changes in dancer_role_info...")

# Проверка наличия и типа dancer_role_info
if 'dancer_role_info' not in globals():
    print("⚠️  Warning: dancer_role_info not found!")
    logging.warning("dancer_role_info not available for change detection")
elif not isinstance(dancer_role_info, pd.DataFrame):
    print(f"⚠️  Warning: dancer_role_info is not a DataFrame (type: {type(dancer_role_info).__name__})!")
    logging.warning(f"dancer_role_info has wrong type: {type(dancer_role_info)}")
elif dancer_role_info.empty:
    print("⚠️  Warning: dancer_role_info DataFrame is empty!")
    logging.warning("dancer_role_info DataFrame is empty")
else:
    try:
        # Путь к предыдущему файлу
        previous_file = 'dancer_role_info.csv'
        
        # Пытаемся загрузить предыдущую версию
        try:
            old_dancer_role_info = pd.read_csv(previous_file)
            print(f"   📂 Loaded previous version: {previous_file}")
            print(f"      Old records: {len(old_dancer_role_info):,}")
            print(f"      New records: {len(dancer_role_info):,}")
            
            # Сравнение
            combined_result_df = pd.concat([old_dancer_role_info, dancer_role_info], ignore_index=True)
            combined_result_df['update_date'] = pd.Timestamp.now().strftime('%Y-%m-%d')
            
            # Находим изменения
            initial_combined = len(combined_result_df)
            combined_result_df = combined_result_df.drop_duplicates(
                subset=combined_result_df.columns.difference(['update_date']),
                keep='last'
            )
            final_combined = len(combined_result_df)
            
            duplicates_removed = initial_combined - final_combined
            new_or_changed = final_combined - len(old_dancer_role_info)
            
            print(f"\n✅ Change detection complete!")
            print(f"   📊 Statistics:")
            print(f"      Total (old + new): {initial_combined:,}")
            print(f"      Duplicates removed: {duplicates_removed:,}")
            print(f"      Final combined: {final_combined:,}")
            print(f"      New or changed: {new_or_changed:,}")
            
            # Сохраняем combined_result_df
            combined_result_df.to_csv('changed_dancer_role_info.csv', index=False)
            print(f"\n   💾 Saved: changed_dancer_role_info.csv ({len(combined_result_df):,} records)")
            
            logging.info(
                f"dancer_role_info change detection: {new_or_changed:,} new/changed, "
                f"{duplicates_removed:,} duplicates removed"
            )
            
        except FileNotFoundError:
            print(f"   ℹ️  No previous version found ({previous_file})")
            print(f"      This appears to be the first run.")
            print(f"      Treating all {len(dancer_role_info):,} records as new.")
            
            # Создаём combined_result_df как копию текущих данных
            combined_result_df = dancer_role_info.copy()
            combined_result_df['update_date'] = pd.Timestamp.now().strftime('%Y-%m-%d')
            combined_result_df.to_csv('changed_dancer_role_info.csv', index=False)
            
            print(f"   💾 Saved: changed_dancer_role_info.csv ({len(combined_result_df):,} records)")
            logging.info(f"First run: all {len(dancer_role_info):,} records saved as new")
        
    except Exception as e:
        print(f"❌ Error in change detection: {type(e).__name__}: {e}")
        logging.error(f"Error in dancer_role_info change detection: {type(e).__name__}: {e}")

print(f"\n✅ Cell 28 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DETECT CHANGES IN DANCERS_POINTS_INFO - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🔍 Detecting changes in dancers_points_info...")

if 'dancers_points_info' not in globals():
    print(f"⚠️  Warning: dancers_points_info not found!")
    logging.warning(f"dancers_points_info not available")
elif not isinstance(dancers_points_info, pd.DataFrame):
    print(f"⚠️  Warning: dancers_points_info is not a DataFrame (type: {type(dancers_points_info).__name__})")
    logging.warning(f"dancers_points_info has wrong type: {type(dancers_points_info)}")
elif dancers_points_info.empty:
    print(f"⚠️  Warning: dancers_points_info DataFrame is empty!")
    logging.warning(f"dancers_points_info DataFrame is empty")
else:
    try:
        previous_file = 'dancers_points_info.csv'
        
        try:
            old_dancers_points_info = pd.read_csv(previous_file)
            print(f"   📂 Loaded previous version: {previous_file}")
            print(f"      Old records: {len(old_dancers_points_info):,}")
            print(f"      New records: {len(dancers_points_info):,}")
            
            combined_points_df = pd.concat([old_dancers_points_info, dancers_points_info], ignore_index=True)
            combined_points_df['update_date'] = pd.Timestamp.now().strftime('%Y-%m-%d')
            
            initial_combined = len(combined_points_df)
            changed_dancers_points_info = combined_points_df.drop_duplicates(
                subset=combined_points_df.columns.difference(['update_date']),
                keep='last'
            )
            final_combined = len(changed_dancers_points_info)
            
            duplicates_removed = initial_combined - final_combined
            new_or_changed = final_combined - len(old_dancers_points_info)
            
            print(f"\n✅ Change detection complete!")
            print(f"   📊 Statistics:")
            print(f"      Duplicates removed: {duplicates_removed:,}")
            print(f"      Final combined: {final_combined:,}")
            print(f"      New or changed: {new_or_changed:,}")
            
            logging.info(f"dancers_points_info: {new_or_changed:,} new/changed")
            
        except FileNotFoundError:
            print(f"   ℹ️  No previous version found. First run.")
            changed_dancers_points_info = dancers_points_info.copy()
            changed_dancers_points_info['update_date'] = pd.Timestamp.now().strftime('%Y-%m-%d')
            logging.info(f"First run: {len(dancers_points_info):,} records")
        
    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {e}")
        logging.error(f"Error in dancers_points_info change detection: {e}")

print(f"\n✅ Cell 29 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# REMOVE DUPLICATES FROM CHANGED DATA - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🔄 Removing duplicates from changed_dancers_points_info...")

if 'changed_dancers_points_info' not in globals():
    print(f"⚠️  Warning: changed_dancers_points_info not found!")
    logging.warning(f"changed_dancers_points_info not available")
elif not isinstance(changed_dancers_points_info, pd.DataFrame):
    print(f"⚠️  Warning: changed_dancers_points_info is not a DataFrame (type: {type(changed_dancers_points_info).__name__})")
    logging.warning(f"changed_dancers_points_info has wrong type: {type(changed_dancers_points_info)}")
elif changed_dancers_points_info.empty:
    print(f"⚠️  Warning: changed_dancers_points_info DataFrame is empty!")
    logging.warning(f"changed_dancers_points_info DataFrame is empty")
else:
    try:
        initial_count = len(changed_dancers_points_info)
        
        changed_dancers_points_info = changed_dancers_points_info.drop_duplicates(
            subset=changed_dancers_points_info.columns.difference(['update_date']),
            keep='last'
        )
        
        final_count = len(changed_dancers_points_info)
        removed = initial_count - final_count
        
        print(f"\n✅ Deduplication complete!")
        print(f"   Initial: {initial_count:,}")
        print(f"   Final: {final_count:,}")
        print(f"   Removed: {removed:,}")
        
        changed_dancers_points_info.to_csv('changed_dancers_points_info.csv', index=False)
        print(f"   💾 Saved: changed_dancers_points_info.csv")
        
        logging.info(f"Deduplicated changed_dancers_points_info: removed {removed:,}")
        
    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {e}")
        logging.error(f"Error in deduplication: {e}")

print(f"\n✅ Cell 30 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIX LOCATION DATA - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🔧 Fixing location data errors...")

if 'location_info' not in globals():
    print(f"⚠️  Warning: location_info not found!")
    logging.warning(f"location_info not available")
elif not isinstance(location_info, pd.DataFrame):
    print(f"⚠️  Warning: location_info is not a DataFrame (type: {type(location_info).__name__})")
    logging.warning(f"location_info has wrong type: {type(location_info)}")
elif location_info.empty:
    print(f"⚠️  Warning: location_info DataFrame is empty!")
    logging.warning(f"location_info DataFrame is empty")
else:
    try:
        # Структурированный список исправлений
        # Формат: (condition_column, condition_value, target_column, new_value)
        location_fixes = [
            ('event_city', 'Morristown', 'event_country', 'United States'),
            ('event_city', 'Morristown', 'event_state', 'New Jersey'),
            ('event_city', 'City: Volna  State: South Federal District  postal 353500 _country: Russia', 'event_city', 'Volna'),
            ('event_country', 'City: Volna  State: South Federal District  postal 353500 _country: Russia', 'event_country', 'Russia'),
            ('event_location', 'City: Volna  State: South Federal District  postal 353500 _country: Russia', 'event_location', 'Volna, Russia'),
            ('event_city', 'Toulouse-Labege', 'event_city', 'Toulouse'),
            ('event_city', 'Overland Park', 'event_state', 'Kansas'),
            ('event_city', 'Raleigh', 'event_state', 'North Carolina'),
            ('event_city', 'New Brunswick', 'event_state', 'New Jersey'),
            ('event_city', 'Atlanta', 'event_state', 'Georgia'),
            ('event_city', 'Denver', 'event_state', 'Colorado'),
            ('event_city', 'Denver', 'event_country', 'United States'),
            ('event_city', 'Orlando', 'event_state', 'Florida'),
            ('event_city', 'Orlando', 'event_country', 'United States'),
            ('event_city', 'Boston', 'event_state', 'Massachusetts'),
            ('event_country', 'Russian', 'event_country', 'Russia'),
            ('event_country', 'USA', 'event_country', 'United States'),
            ('event_city', 'WILMINGTON', 'event_state', 'Delaware'),
            ('event_city', 'WILMINGTON', 'event_country', 'United States'),
            ('event_city', 'WILMINGTON', 'event_city', 'Wilmington'),
            ('event_city', 'Perth', 'event_country', 'Australia'),
            ('event_country', 'Russian ', 'event_country', 'Russia'),
            ('event_city', 'Baton Rouge', 'event_country', 'United States'),
            ('event_city', 'Baton Rouge', 'event_state', 'Louisiana'),
            ('event_city', 'Overland Park', 'event_country', 'United States'),
            ('event_city', 'Detroit', 'event_state', 'Michigan'),
            ('event_city', 'San Jose', 'event_state', 'California'),
            ('event_country', ' Hungary', 'event_country', 'Hungary'),
            ('event_country', ' Italy', 'event_country', 'Italy'),
            ('event_city', 'Cleveland', 'event_state', 'Ohio'),
            ('event_city', 'Cleveland', 'event_country', 'United States'),
            ('event_city', 'LYON France', 'event_city', 'Lyon'),
            ('event_city', 'LYON', 'event_city', 'Lyon'),
            ('event_city', 'Irvine', 'event_state', 'California'),
            ('event_city', 'Irvine', 'event_country', 'United States'),
            ('event_city', 'Paris', 'event_country', 'France'),
            ('event_city', 'South Lake Tahoe', 'event_state', 'Nevada'),
            ('event_city', 'Nashville', 'event_state', 'Tennessee'),
            ('event_city', 'Austin', 'event_state', 'Texas'),
            ('event_city', 'Austin', 'event_country', 'United States'),
            ('event_city', 'Fort Lauderdale', 'event_state', 'Florida'),
            ('event_city', 'South Lake Tahoe', 'event_state', 'Washington'),
            ('event_city', 'Sacramento', 'event_state', 'California'),
            ('event_city', 'Irvine Orange County', 'event_state', 'California'),
            ('event_city', 'Irvine Orange County', 'event_country', 'United States'),
            ('event_city', 'Irvine Orange County', 'event_city', 'Irvine'),
            ('event_city', 'Dallas Ft. Worth', 'event_state', 'Texas'),
            ('event_city', 'Dallas Ft. Worth', 'event_city', 'Dallas'),
            ('event_city', 'Seattle', 'event_state', 'Washington'),
            ('event_city', 'Louisville', 'event_state', 'Kentucky'),
            ('event_city', 'Zurich', 'event_country', 'Switzerland'),
            ('event_city', 'Live Oak', 'event_state', 'Texas'),
            ('event_city', 'Live Oak', 'event_country', 'United States'),
            ('event_country', 'South Korea', 'event_country', 'Republic of Korea'),
            ('event_city', 'SEOUL', 'event_country', 'Republic of Korea'),
            ('event_city', 'SEOUL', 'event_city', 'Seoul'),
            ('event_city', 'Stockholm', 'event_country', 'Sweden'),
            ('event_city', 'Windsor Locks', 'event_country', 'United States'),
            ('event_city', 'Windsor Locks', 'event_state', 'Connecticut'),
            ('event_city', 'Rust', 'event_country', 'Germany'),
            ('event_city', 'Houston', 'event_country', 'United States'),
            ('event_city', 'Houston', 'event_state', 'Texas'),
            ('event_city', 'NANTES', 'event_city', 'Nantes'),
            ('event_country', ' Singapore', 'event_country', 'Singapore'),
            ('event_city', 'PARIS', 'event_city', 'Paris'),
            ('event_country', 'CANADA', 'event_country', 'Canada'),
            ('event_country', ' Russia', 'event_country', 'Russia'),
            ('event_country', ' Germany', 'event_country', 'Germany'),
            ('event_country', ' Austria', 'event_country', 'Austria'),
            ('event_country', 'Deutschland', 'event_country', 'Germany'),
            ('event_city', 'Annecy ', 'event_city', 'Annecy'),
            ('event_city', 'Annecy', 'event_country', 'France'),
            ('event_city', 'Boston Club', 'event_city', 'Düsseldorf'),
            ('event_country', 'Czechia', 'event_country', 'Czech Republic'),
            ('event_city', 'Washington', 'event_state', 'District of Columbia'),
            ('event_city', 'Phoenix', 'event_state', 'Arizona'),
            ('event_city', 'Lancaster', 'event_state', 'California'),
            ('event_city', 'Dallas', 'event_state', 'Texas'),
            ('event_city', 'Chicago', 'event_state', 'Illinois'),
            ('event_city', 'Tampa Bay', 'event_state', 'Florida'),
            ('event_city', 'Los Angels', 'event_state', 'California'),
            ('event_city', 'Tulsa', 'event_state', 'Oklahoma'),
            ('event_city', 'Ashland', 'event_state', 'Oregon'),
            ('event_city', 'Portland', 'event_state', 'Oregon'),
            ('event_city', 'Hartfoed', 'event_state', 'Connecticut'),
            ('event_city', 'Fort Wayne', 'event_state', 'Indiana'),
            ('event_city', 'Barcelona', 'event_country', 'Spain'),
            ('event_city', 'San Antonio', 'event_state', 'Texas'),
            ('event_city', 'Greenville', 'event_state', 'South Carolina'),
            ('event_city', 'Nanaimo (Vancouver is the major in the province)', 'event_city', 'Nanaimo'),
            ('event_city', 'Los Angels', 'event_city', 'Los Angeles'),
            ('event_city', 'Tampa Bay', 'event_city', 'Tampa'),
            ('event_city', 'Ft. Lauderdale', 'event_city', 'Fort Lauderdale'),
            ('event_city', 'Fort Lauderdale', 'event_state', 'Florida'),
            ('event_city', 'St.Petersburg', 'event_city', 'St. Petersburg'),
            ('event_city', 'Washington DC', 'event_state', 'District of Columbia'),
            ('event_city', 'Washington DC', 'event_city', 'Washington'),
            ('event_city', 'Anaheim/Garden Grove', 'event_state', 'California'),
            ('event_city', 'Anaheim/Garden Grove', 'event_city', 'Anaheim'),
            ('event_city', 'San Francisco', 'event_state', 'California'),
            ('event_city', 'South Lake Tahoe', 'event_state', 'California'),
            ('event_city', 'N. Myrtle Beach', 'event_city', 'North Myrtle Beach'),
            ('event_country', ' France', 'event_country', 'France'),
            ('event_country', 'The Netherlands', 'event_country', 'Netherlands'),
            ('event_country', ' New Zealand', 'event_country', 'New Zealand'),
            ('event_country', ' Portugal', 'event_country', 'Portugal'),
            ('event_country', ' Malaysia', 'event_country', 'Malaysia'),
            ('event_city', 'Hartfoed', 'event_city', 'Hartford'),
            ('event_city', 'EDINBURGH', 'event_city', 'Edinburgh'),
            ('event_country', 'Edinburgh', 'event_country', 'United Kingdom'),
            ('event_country', ' Slovenia', 'event_country', 'Slovenia'),
            ('event_country', ' Bulgaria', 'event_country', 'Bulgaria'),
            ('event_city', 'Jacksonville', 'event_state', 'Florida'),
            ('event_city', 'St. Louis', 'event_state', 'Missouri'),
            ('event_country', 'SCOTLAND', 'event_country', 'United Kingdom'),
            ('event_city', 'Wailea', 'event_state', 'Hawaii'),
            ('event_city', 'Huntsville', 'event_state', 'Alabama'),
            ('event_city', 'Cincinnati', 'event_state', 'Ohio'),
        ]
        
        initial_records = len(location_info)
        fixes_applied = 0
        
        print(f"   Total location fixes to apply: {len(location_fixes):,}")
        
        # Применяем все исправления
        for cond_col, cond_val, target_col, new_val in location_fixes:
            if cond_col in location_info.columns and target_col in location_info.columns:
                mask = location_info[cond_col] == cond_val
                affected = mask.sum()
                
                if affected > 0:
                    location_info.loc[mask, target_col] = new_val
                    fixes_applied += affected
        
        print(f"\n✅ Location fixes applied!")
        print(f"   📊 Statistics:")
        print(f"      Total records: {initial_records:,}")
        print(f"      Records affected: {fixes_applied:,}")
        print(f"      Fix rules applied: {len(location_fixes):,}")
        
        logging.info(f"Applied {len(location_fixes):,} location fix rules, affected {fixes_applied:,} records")
        
    except Exception as e:
        print(f"❌ Error fixing locations: {type(e).__name__}: {e}")
        logging.error(f"Error in location fixes: {e}")

print(f"\n✅ Cell 31 complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIX DALLAS COORDINATES - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n🔧 Fixing Dallas coordinates...")

if 'location_info' not in globals():
    print(f"⚠️  Warning: location_info not found!")
    logging.warning(f"location_info not available")
elif not isinstance(location_info, pd.DataFrame):
    print(f"⚠️  Warning: location_info is not a DataFrame (type: {type(location_info).__name__})")
    logging.warning(f"location_info has wrong type: {type(location_info)}")
elif location_info.empty:
    print(f"⚠️  Warning: location_info DataFrame is empty!")
    logging.warning(f"location_info DataFrame is empty")
else:
    try:
        mask = location_info['event_city'] == 'Dallas'
        affected = mask.sum()
        
        if affected > 0:
            location_info.loc[mask, 'latitude'] = 32.7831
            location_info.loc[mask, 'longitude'] = -96.8067
            print(f"✅ Fixed coordinates for {affected:,} Dallas records")
            print(f"   Lat: 32.7831, Lng: -96.8067")
            logging.info(f"Fixed Dallas coordinates for {affected} records")
        else:
            print("ℹ️  No Dallas records found")
    except Exception as e:
        print(f"❌ Error: {e}")
        logging.error(f"Error fixing Dallas coordinates: {e}")

print(f"✅ Cell 32 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EXPORT DATA TO CSV - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n💾 Exporting data to CSV files...")

# ═══════════════════════════════════════════════════════════════════
# DATA QUALITY PREPROCESSING - BEFORE EXPORT
# ═══════════════════════════════════════════════════════════════════

if DATA_PREPROCESSING_AVAILABLE:
    print("\n🔄 Applying data quality improvements before export...")
    
    try:
        # Собираем все данные для предобработки
        data_to_process = {}
        
        if 'location_info' in globals() and isinstance(location_info, pd.DataFrame):
            data_to_process['location_info'] = location_info.copy()
        if 'dancer_role_info' in globals() and isinstance(dancer_role_info, pd.DataFrame):
            data_to_process['dancer_role_info'] = dancer_role_info.copy()
        if 'dancers_results_info' in globals() and isinstance(dancers_results_info, pd.DataFrame):
            data_to_process['dancers_results_info'] = dancers_results_info.copy()
        
        if data_to_process:
            # Применяем стандартизацию форматов
            processed_data = standardize_formats(data_to_process)
            
            # Обновляем глобальные переменные
            for key, df in processed_data.items():
                if key in globals():
                    globals()[key] = df
                    print(f"   ✅ {key}: formats standardized")
            
            print("   ✅ Data quality improvements applied")
        else:
            print("   ℹ️  No data available for preprocessing")
            
    except Exception as e:
        print(f"   ⚠️  Warning: Data quality preprocessing failed: {e}")
        logging.warning(f"Data quality preprocessing failed: {e}")
else:
    print("ℹ️  Data preprocessing not available, exporting without quality improvements")

export_tasks = [
    ('dancer_role_info', 'dancer_role_info.csv'),
    ('dancers_points_info', 'dancers_points_info.csv'),
    ('dancers_results_info', 'dancers_results_info.csv'),
    ('location_info', 'location_info.csv'),
]

exported_count = 0
failed_count = 0

for df_name, filename in export_tasks:
    if df_name in globals():
        df = globals()[df_name].copy()  # Используем копию, чтобы не изменять оригинал
        if not df.empty:
            try:
                # Добавляем update_date для dancer_role_info и dancers_points_info
                if df_name in ['dancer_role_info', 'dancers_points_info']:
                    if 'update_date' not in df.columns:
                        df['update_date'] = pd.Timestamp.now().strftime('%Y-%m-%d')
                        print(f"   ℹ️  Added update_date to {filename}")
                
                df.to_csv(filename, index=False)
                records = len(df)
                size_mb = os.path.getsize(filename) / (1024 * 1024)
                print(f"   ✅ {filename}: {records:,} records ({size_mb:.2f} MB)")
                exported_count += 1
            except Exception as e:
                print(f"   ❌ {filename}: Error - {e}")
                failed_count += 1
        else:
            print(f"   ⚠️  {df_name} is empty, skipping")
    else:
        print(f"   ⚠️  {df_name} not found, skipping")

print(f"\n✅ Export complete! {exported_count} files exported, {failed_count} failed")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EXTRACT EVENTS LIST - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n📋 Extracting events list...")

if 'soup_list' not in globals() or not soup_list:
    print("⚠️  Warning: soup_list not found or empty!")
    logging.warning("soup_list not available for event extraction")
else:
    try:
        events_records = []
        seen_keys = set()

        def iter_competitions(dancer_block):
            if not isinstance(dancer_block, dict):
                return
            placements = dancer_block.get('placements', {})
            if isinstance(placements, dict):
                for dance_data in placements.values():
                    if not isinstance(dance_data, dict):
                        continue
                    for level_data in dance_data.values():
                        if not isinstance(level_data, dict):
                            continue
                        competitions = level_data.get('competitions', [])
                        if isinstance(competitions, list):
                            for competition in competitions:
                                yield competition

        for soup in soup_list:
            for role_key in ('leader', 'follower'):
                for competition in iter_competitions(soup.get(role_key, {})):
                    if not isinstance(competition, dict):
                        continue

                    event_data = competition.get('event', {})
                    if not isinstance(event_data, dict):
                        continue

                    event_id = event_data.get('id')
                    event_name = str(event_data.get('name', '')).strip()
                    event_location = str(event_data.get('location', '')).strip()
                    event_url = str(event_data.get('url', '')).strip()
                    event_date = str(event_data.get('date', '')).strip()

                    # Нормализуем идентификатор
                    if pd.notna(event_id):
                        try:
                            event_id = int(event_id)
                        except (TypeError, ValueError):
                            event_id = str(event_id).strip()
                    else:
                        event_id = None

                    key = (event_id, event_name, event_location, event_date)
                    if key in seen_keys:
                        continue
                    seen_keys.add(key)

                    if not any([event_name, event_location, event_date]):
                        continue

                    events_records.append({
                        'id': event_id,
                        'name': event_name or 'Unknown Event',
                        'location': event_location,
                        'url': event_url,
                        'date': event_date
                    })

        events_wsdc = pd.DataFrame(events_records, columns=['id', 'name', 'location', 'url', 'date'])

        if events_wsdc.empty:
            print("⚠️  No events detected in soup_list")
            logging.warning("No events extracted from soup_list")
        else:
            events_wsdc.sort_values(by=['id', 'date', 'name'], inplace=True, na_position='last')
            events_wsdc.reset_index(drop=True, inplace=True)
            events_wsdc.to_csv('events_wsdc.csv', index=False)

            sample_count = min(5, len(events_wsdc))
            print(f"✅ Extracted {len(events_wsdc):,} unique events")
            print(f"   📄 Saved to events_wsdc.csv")
            if sample_count:
                print("   🔍 Sample:")
                for _, row in events_wsdc.head(sample_count).iterrows():
                    print(f"      • {row['date']} — {row['name']} ({row['location']})")

            logging.info(f"Extracted {len(events_wsdc)} events from soup_list")

    except Exception as e:
        print(f"❌ Error: {e}")
        logging.error(f"Error extracting events: {e}")

print(f"✅ Cell 34 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DIVISIONAL TRANSITIONS TABLE - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════

print("\n📊 Building divisional transitions table...")

from pathlib import Path
from datetime import datetime

DIVISION_MAP = {
    'NEW': 'Newcomer',
    'NOV': 'Novice',
    'INT': 'Intermediate',
    'ADV': 'Advanced',
    'ALS': 'All-Star',
    'ALL': 'All-Star',
    'CHMP': 'Champion',
    'N/A': None,
    'NA': None,
    'NONE': None,
    '': None
}

def normalize_division(value):
    if pd.isna(value):
        return None
    val = str(value).strip()
    if not val or val.lower() in {"n/a", "na", "none", "null"}:
        return None
    return DIVISION_MAP.get(val.upper(), val.title())

try:
    current_path = Path('dancer_role_info.csv')
    backup_candidates = sorted(Path('.').glob('dancer_role_info_BACKUP_*.csv'))
    previous_path = backup_candidates[-1] if backup_candidates else None

    if not current_path.exists():
        raise FileNotFoundError("dancer_role_info.csv not found")

    df_current = pd.read_csv(current_path)
    print(f"   Loaded current snapshot: {current_path.name} ({len(df_current):,} records)")

    if previous_path and previous_path.exists():
        df_previous = pd.read_csv(previous_path)
        print(f"   Loaded previous snapshot: {previous_path.name} ({len(df_previous):,} records)")
    else:
        df_previous = pd.DataFrame()
        print("   ⚠️ Previous snapshot not found. Transition analysis will be skipped.")

    df_transitions = pd.DataFrame()

    if df_previous.empty:
        logging.warning("Transition analysis skipped: previous snapshot unavailable.")
    else:
        merge_keys = ['dancer_id', 'dancer_name']
        df_current['dancer_id'] = pd.to_numeric(df_current['dancer_id'], errors='coerce').astype('Int64')
        df_previous['dancer_id'] = pd.to_numeric(df_previous['dancer_id'], errors='coerce').astype('Int64')

        merged = df_current.merge(
            df_previous,
            on=merge_keys,
            how='left',
            suffixes=('_curr', '_prev')
        )

        if 'update_date' in df_current.columns:
            update_values = df_current['update_date'].dropna().unique()
            current_update_date = update_values[0] if len(update_values) else datetime.now().strftime('%Y-%m-%d')
        else:
            current_update_date = datetime.now().strftime('%Y-%m-%d')

        transitions = []
        role_specs = [
            ('dominate', 'Dominant'),
            ('non_dominate', 'Non-Dominant')
        ]

        for _, row in merged.iterrows():
            dancer_id = row.get('dancer_id')
            if pd.isna(dancer_id):
                continue

            dancer_id = int(dancer_id)
            dancer_name = row.get('dancer_name', '')

            for prefix, _ in role_specs:
                role_value = row.get(f'{prefix}_role_curr')
                if pd.isna(role_value) or not str(role_value).strip():
                    continue

                dancer_role = str(role_value).strip()

                for field, label in [('required', 'required'), ('allowed', 'allowed')]:
                    prev_val = normalize_division(row.get(f'{prefix}_{field}_prev'))
                    curr_val = normalize_division(row.get(f'{prefix}_{field}_curr'))

                    if prev_val == curr_val:
                        continue

                    if prev_val is None and curr_val is None:
                        continue

                    transitions.append({
                        'Update Date': current_update_date,
                        'Previous Division': prev_val or '—',
                        'Currently Division': curr_val or '—',
                        'Transition Type': label,
                        'Dancer Role': dancer_role,
                        'Dancer ID': dancer_id,
                        'Dancer Name': dancer_name
                    })

        df_transitions = pd.DataFrame(transitions)

        if df_transitions.empty:
            print("   ℹ️ No divisional transitions detected.")
            logging.info("No divisional transitions detected.")
        else:
            df_transitions.sort_values(
                ['Update Date', 'Dancer ID', 'Dancer Role', 'Transition Type'],
                inplace=True
            )
            df_transitions.to_csv('divisional_transitions.csv', index=False)

            print(f"\n✅ Divisional transitions analysis complete!")
            print(f"   📊 Total transitions found: {len(df_transitions):,}")
            print(f"   💾 Saved: divisional_transitions.csv")

            transition_types = (
                df_transitions
                .groupby(['Previous Division', 'Currently Division'])
                .size()
                .sort_values(ascending=False)
            )

            print(f"\n   📈 Top 5 transition types:")
            for (from_div, to_div), count in transition_types.head(5).items():
                print(f"      {from_div} → {to_div}: {count}")

            logging.info(f"Built divisional transitions table: {len(df_transitions)} transitions")

    if df_transitions.empty:
        print("   ⚠️ Transition file not created due to lack of changes.")

except FileNotFoundError as err:
    print(f"⚠️  {err}")
except Exception as e:
    print(f"❌ Error: {e}")
    logging.error(f"Error building transitions: {e}")

print(f"\n✅ Cell 44 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# AGGREGATE DIVISIONAL STRUCTURE - АГРЕГАТЫ (СЛЕПКИ)
# ═══════════════════════════════════════════════════════════════════
#
# 🎯 НАЗНАЧЕНИЕ: Создание агрегированных слепков структуры дивизионов
#                для ВСЕХ ролей (dominate и non_dominate)
#
# 📋 ПРОЦЕСС:
#    1. Читает changed_dancer_role_info.csv
#    2. Обрабатывает dominate_role (dominate_required, dominate_allowed)
#    3. Обрабатывает non_dominate_role (non_dominate_required, non_dominate_allowed)
#    4. Преобразует в длинный формат (melt)
#    5. Группирует и АГРЕГИРУЕТ по division, role, type → count_dancer
#    6. Сохраняет АГРЕГАТЫ (слепки) - количество танцоров в каждом дивизионе
#    7. Объединяет с предыдущими слепками
#
# 📊 РЕЗУЛЬТАТ: Агрегированные данные (update_date, division, role, type, count_dancer)
#               БЕЗ детальных записей с dancer_id
# ═══════════════════════════════════════════════════════════════════

import os
import logging

print("\n📊 Aggregating divisional structure (all roles)...")

try:
    aggregated_file = "divisional_structure.csv"
    
    # ═══════════════════════════════════════════════════════════════
    # 1. Загрузка данных
    # ═══════════════════════════════════════════════════════════════
    
    from datetime import datetime
    
    # Пробуем загрузить changed_dancer_role_info.csv с датами
    if os.path.exists('changed_dancer_role_info.csv'):
        df_ds = pd.read_csv('changed_dancer_role_info.csv', low_memory=False)
        # Проверяем, есть ли записи с датами
        if 'update_date' in df_ds.columns and df_ds['update_date'].notna().any():
            print(f"   📥 Loaded: changed_dancer_role_info.csv ({len(df_ds):,} records)")
            print(f"   📅 Records with dates: {df_ds['update_date'].notna().sum()}")
        else:
            # Нет дат - используем текущий snapshot
            print(f"   ⚠️  changed_dancer_role_info.csv has no dates, using current snapshot")
            if os.path.exists('dancer_role_info.csv'):
                df_ds = pd.read_csv('dancer_role_info.csv', low_memory=False)
                # Добавляем текущую дату для создания слепка
                today = datetime.now().strftime('%Y-%m-%d')
                df_ds['update_date'] = today
                print(f"   📥 Loaded: dancer_role_info.csv ({len(df_ds):,} records)")
                print(f"   📅 Added update_date: {today}")
            else:
                raise FileNotFoundError("Neither changed_dancer_role_info.csv nor dancer_role_info.csv found")
    else:
        # Используем текущий snapshot
        if os.path.exists('dancer_role_info.csv'):
            df_ds = pd.read_csv('dancer_role_info.csv', low_memory=False)
            today = datetime.now().strftime('%Y-%m-%d')
            df_ds['update_date'] = today
            print(f"   📥 Loaded: dancer_role_info.csv ({len(df_ds):,} records)")
            print(f"   📅 Added update_date: {today}")
        else:
            raise FileNotFoundError("dancer_role_info.csv not found")
    
    initial_count = len(df_ds)
    
    # ═══════════════════════════════════════════════════════════════
    # 2. Обработка dominate_role
    # ═══════════════════════════════════════════════════════════════
    
    df_dominate = df_ds[df_ds["dominate_role"].notna()].copy()
    
    if len(df_dominate) > 0:
        df_dominate_melted = pd.melt(
            df_dominate,
            id_vars=["dancer_id", "update_date", "dancer_name", "dominate_role", 
                     "non_dominate_role", "non_dominate_required", "non_dominate_allowed",
                     "non_dominate_recommended", "non_dominate_role_highest_level_points",
                     "non_dominate_role_highest_level"],
            value_vars=["dominate_required", "dominate_allowed"],
            var_name="type_options",
            value_name="division"
        )
        
        def extract_type(row):
            return "required" if "required" in row["type_options"] else "allowed"
        
        df_dominate_melted["type"] = df_dominate_melted.apply(extract_type, axis=1)
        # ВАЖНО: переименовываем ПЕРЕД фильтрацией
        df_dominate_melted.rename(columns={"dominate_role": "role"}, inplace=True)
        df_dominate_melted = df_dominate_melted[df_dominate_melted["division"].notna()]
        # Удаляем записи без update_date (они не нужны для слепков)
        df_dominate_melted = df_dominate_melted[df_dominate_melted["update_date"].notna()]
        
        print(f"   ✓ Processed dominate_role: {len(df_dominate_melted):,} records")
    else:
        df_dominate_melted = pd.DataFrame()
        print(f"   ⚠️  No records with dominate_role found")
    
    # ═══════════════════════════════════════════════════════════════
    # 3. Обработка non_dominate_role
    # ═══════════════════════════════════════════════════════════════
    
    df_non_dominate = df_ds[df_ds["non_dominate_role"].notna()].copy()
    
    if len(df_non_dominate) > 0:
        df_non_dominate_melted = pd.melt(
            df_non_dominate,
            id_vars=["dancer_id", "update_date", "dancer_name", "dominate_role",
                     "dominate_required", "dominate_allowed", "non_dominate_role",
                     "non_dominate_recommended", "non_dominate_role_highest_level_points",
                     "non_dominate_role_highest_level"],
            value_vars=["non_dominate_required", "non_dominate_allowed"],
            var_name="type_options",
            value_name="division"
        )
        
        df_non_dominate_melted["type"] = df_non_dominate_melted.apply(extract_type, axis=1)
        df_non_dominate_melted.rename(columns={"non_dominate_role": "role"}, inplace=True)
        df_non_dominate_melted = df_non_dominate_melted[df_non_dominate_melted["division"].notna()]
        # Удаляем записи без update_date (они не нужны для слепков)
        df_non_dominate_melted = df_non_dominate_melted[df_non_dominate_melted["update_date"].notna()]
        
        print(f"   ✓ Processed non_dominate_role: {len(df_non_dominate_melted):,} records")
    else:
        df_non_dominate_melted = pd.DataFrame()
        print(f"   ⚠️  No records with non_dominate_role found")
    
    # ═══════════════════════════════════════════════════════════════
    # 4. Объединение всех данных
    # ═══════════════════════════════════════════════════════════════
    
    if len(df_dominate_melted) > 0 and len(df_non_dominate_melted) > 0:
        # Выбираем общие колонки
        common_cols = ["dancer_id", "update_date", "division", "role", "type", "dancer_name",
                       "dominate_role", "dominate_required", "dominate_allowed",
                       "non_dominate_role", "non_dominate_required", "non_dominate_allowed",
                       "non_dominate_recommended", "non_dominate_role_highest_level_points",
                       "non_dominate_role_highest_level"]
        
        df_dominate_melted = df_dominate_melted[[c for c in common_cols if c in df_dominate_melted.columns]]
        df_non_dominate_melted = df_non_dominate_melted[[c for c in common_cols if c in df_non_dominate_melted.columns]]
        
        df_melted = pd.concat([df_dominate_melted, df_non_dominate_melted], ignore_index=True)
    elif len(df_dominate_melted) > 0:
        df_melted = df_dominate_melted
    elif len(df_non_dominate_melted) > 0:
        df_melted = df_non_dominate_melted
    else:
        df_melted = pd.DataFrame()
    
    if len(df_melted) == 0:
        print(f"\n⚠️  Warning: No data to aggregate!")
        logging.warning("No data to aggregate in divisional structure")
    else:
        print(f"   ✓ Total melted records: {len(df_melted):,}")
        
        # ═══════════════════════════════════════════════════════════════
        # 5. АГРЕГАЦИЯ: Группировка и подсчет количества танцоров
        # ═══════════════════════════════════════════════════════════════
        
        # Удаляем временную колонку type_options, оставляем только type
        if 'type_options' in df_melted.columns:
            df_melted = df_melted.drop(columns=['type_options'])
        
        # АГРЕГАЦИЯ: группируем по (update_date, division, role, type) и считаем количество танцоров
        new_aggregated_df = (
            df_melted.groupby(["update_date", "division", "role", "type"])
            .agg(count_dancer=("dancer_id", "count"))
            .reset_index()
        )
        
        # Добавляем остальные колонки (пустые для агрегатов)
        additional_cols = ["dancer_id", "dancer_name", "dominate_role", "dominate_required", 
                          "dominate_allowed", "non_dominate_role", "non_dominate_required", 
                          "non_dominate_allowed", "non_dominate_recommended", 
                          "non_dominate_role_highest_level_points", "non_dominate_role_highest_level"]
        
        for col in additional_cols:
            new_aggregated_df[col] = None
        
        # Порядок колонок как в divisional_structure_only_dominate_role.csv
        final_cols = ["update_date", "division", "role", "type", "count_dancer"] + additional_cols
        new_aggregated_df = new_aggregated_df[final_cols]
        
        new_agg_count = len(new_aggregated_df)
        print(f"   ✓ Aggregated: {new_agg_count:,} unique combinations (snapshots)")
        
        # ═══════════════════════════════════════════════════════════════
        # 6. Объединение с предыдущими данными
        # ═══════════════════════════════════════════════════════════════
        
        # ═══════════════════════════════════════════════════════════════
        # 6. Объединение с предыдущими слепками
        # ═══════════════════════════════════════════════════════════════
        
        if os.path.exists(aggregated_file):
            existing_data = pd.read_csv(aggregated_file, low_memory=False)
            existing_count = len(existing_data)
            print(f"   📥 Loaded previous: {aggregated_file} ({existing_count:,} snapshots)")
            
            # Убеждаемся, что структура совпадает
            if 'type_options' in existing_data.columns and 'type' not in existing_data.columns:
                # Старая структура - конвертируем
                def extract_type_old(row):
                    return "required" if "required" in str(row.get("type_options", "")) else "allowed"
                existing_data["type"] = existing_data.apply(extract_type_old, axis=1)
                if 'type_options' in existing_data.columns:
                    existing_data = existing_data.drop(columns=['type_options'])
            
            # Добавляем отсутствующие колонки в существующие данные
            for col in final_cols:
                if col not in existing_data.columns:
                    existing_data[col] = None
            
            # Приводим к одному порядку колонок
            existing_data = existing_data[final_cols]
            new_aggregated_df = new_aggregated_df[final_cols]
            
            # Удаляем старые слепки с теми же датами перед добавлением новых
            if 'update_date' in new_aggregated_df.columns and new_aggregated_df['update_date'].notna().any():
                dates_to_replace = new_aggregated_df['update_date'].dropna().unique()
                print(f"   📅 Dates in new snapshots: {len(dates_to_replace)} unique dates")
                # Удаляем старые записи с этими датами
                existing_data = existing_data[
                    ~existing_data['update_date'].isin(dates_to_replace)
                ]
                removed_old = existing_count - len(existing_data)
                if removed_old > 0:
                    print(f"   🗑️  Removed {removed_old:,} old snapshots with same dates")
            
            # Объединяем агрегаты
            combined_data = pd.concat([existing_data, new_aggregated_df], ignore_index=True)
            combined_data = combined_data.drop_duplicates(
                subset=["update_date", "division", "role", "type"],
                keep='last'
            )
            
            final_count = len(combined_data)
            added = final_count - len(existing_data)
            print(f"   ✓ Combined: {final_count:,} total snapshots (+{added:,} new)")
        else:
            print(f"   ℹ️  No previous data found, creating new file")
            combined_data = new_aggregated_df
            final_count = len(combined_data)
        
        # ═══════════════════════════════════════════════════════════════
        # 7. Сохранение
        # ═══════════════════════════════════════════════════════════════
        
        combined_data.to_csv(aggregated_file, index=False)
        
        print(f"\n✅ Divisional structure aggregated!")
        print(f"   📊 Total records: {final_count:,}")
        print(f"   💾 Saved: {aggregated_file}")
        
        # Показываем примеры агрегатов
        if final_count > 0:
            print(f"\n   📋 Sample aggregated snapshots:")
            sample = combined_data.head(5)
            for _, row in sample.iterrows():
                date_str = str(row['update_date']) if pd.notna(row['update_date']) else "N/A"
                div_str = str(row['division']) if pd.notna(row['division']) else "N/A"
                role_str = str(row['role']) if pd.notna(row['role']) else "N/A"
                type_str = str(row['type']) if pd.notna(row['type']) else "N/A"
                count = row['count_dancer'] if pd.notna(row['count_dancer']) else 0
                print(f"      {date_str} | {div_str:20s} | {role_str:8s} | {type_str:8s} | {count:,.0f} dancers")
        
        logging.info(f"Aggregated divisional structure: {final_count} total records")
    
except Exception as e:
    print(f"❌ Error: {e}")
    logging.error(f"Error aggregating divisional structure: {e}")
    import traceback
    traceback.print_exc()

print(f"\n✅ Cell 45 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DIVISIONAL STRUCTURE (DOMINATE ROLE ONLY) - OPTIMIZED VERSION
# ═══════════════════════════════════════════════════════════════════
#
# 🎯 НАЗНАЧЕНИЕ: Агрегация данных по дивизионам только для основной роли
#
# 📋 ПРОЦЕСС:
#    1. Читает changed_dancer_role_info.csv
#    2. Фильтрует по наличию dominate_role
#    3. Преобразует в длинный формат (melt)
#    4. Группирует и агрегирует по division, role, type
#    5. Объединяет с предыдущими данными
# ═══════════════════════════════════════════════════════════════════

import os
import logging

print("\n📊 Aggregating divisional structure (dominate role only)...")

try:
    aggregated_file = "divisional_structure_only_dominate_role.csv"
    
    # ═══════════════════════════════════════════════════════════════
    # 1. Загрузка данных
    # ═══════════════════════════════════════════════════════════════
    
    df_ds = pd.read_csv('changed_dancer_role_info.csv')
    initial_count = len(df_ds)
    print(f"   📥 Loaded: changed_dancer_role_info.csv ({initial_count:,} records)")
    
    # ═══════════════════════════════════════════════════════════════
    # 2. Фильтрация по наличию dominate_role
    # ═══════════════════════════════════════════════════════════════
    
    df_ds = df_ds[df_ds["dominate_role"].notna()]
    filtered_count = len(df_ds)
    print(f"   ✓ Filtered by dominate_role: {filtered_count:,} records ({filtered_count/initial_count*100:.1f}%)")
    
    if filtered_count == 0:
        print(f"\n⚠️  Warning: No records with dominate_role found!")
        logging.warning("No records with dominate_role in changed_dancer_role_info")
    else:
        # ═══════════════════════════════════════════════════════════════
        # 3. Преобразование в длинный формат (melt)
        # ═══════════════════════════════════════════════════════════════
        
        df_melted = pd.melt(
            df_ds,
            id_vars=["dancer_id", "update_date", "dominate_role"],
            value_vars=["dominate_required", "dominate_allowed"],
            var_name="type_options",
            value_name="division"
        )
        
        print(f"   ✓ Melted to long format: {len(df_melted):,} records")
        
        # ═══════════════════════════════════════════════════════════════
        # 4. Определение типа (required/allowed)
        # ═══════════════════════════════════════════════════════════════
        
        def extract_type(row):
            return "required" if "required" in row["type_options"] else "allowed"
        
        df_melted["type"] = df_melted.apply(extract_type, axis=1)
        df_melted.rename(columns={"dominate_role": "role"}, inplace=True)
        
        # ═══════════════════════════════════════════════════════════════
        # 5. Группировка и агрегация
        # ═══════════════════════════════════════════════════════════════
        
        new_aggregated_df = (
            df_melted.groupby(["update_date", "division", "role", "type"])
            .agg(count_dancer=("dancer_id", "count"))
            .reset_index()
        )
        
        new_agg_count = len(new_aggregated_df)
        print(f"   ✓ Aggregated: {new_agg_count:,} unique combinations")
        
        # ═══════════════════════════════════════════════════════════════
        # 6. Объединение с предыдущими данными
        # ═══════════════════════════════════════════════════════════════
        
        if os.path.exists(aggregated_file):
            existing_data = pd.read_csv(aggregated_file)
            existing_count = len(existing_data)
            print(f"   📥 Loaded previous: {aggregated_file} ({existing_count:,} records)")
            
            # Удаляем старые слепки с теми же датами перед добавлением новых
            if 'update_date' in new_aggregated_df.columns and new_aggregated_df['update_date'].notna().any():
                dates_to_replace = new_aggregated_df['update_date'].dropna().unique()
                print(f"   📅 Dates in new snapshots: {len(dates_to_replace)} unique dates")
                # Удаляем старые записи с этими датами
                existing_data = existing_data[
                    ~existing_data['update_date'].isin(dates_to_replace)
                ]
                removed_old = existing_count - len(existing_data)
                if removed_old > 0:
                    print(f"   🗑️  Removed {removed_old:,} old snapshots with same dates")
            
            # Объединяем агрегаты
            combined_data = pd.concat([existing_data, new_aggregated_df], ignore_index=True)
            combined_data = combined_data.drop_duplicates(
                subset=["update_date", "division", "role", "type"],
                keep='last'
            )
            
            final_count = len(combined_data)
            added = final_count - existing_count
            print(f"   ✓ Combined: {final_count:,} total (+{added:,} new)")
        else:
            print(f"   ℹ️  No previous data found, creating new file")
            combined_data = new_aggregated_df
            final_count = len(combined_data)
        
        # ═══════════════════════════════════════════════════════════════
        # 7. Сохранение
        # ═══════════════════════════════════════════════════════════════
        
        combined_data.to_csv(aggregated_file, index=False)
        
        print(f"\n✅ Divisional structure (dominate role) aggregated!")
        print(f"   📊 Total records: {final_count:,}")
        print(f"   💾 Saved: {aggregated_file}")
        
        # Примеры данных
        print(f"\n   📋 Sample aggregated data:")
        for idx, row in combined_data.head(5).iterrows():
            print(f"      {row['update_date']} | {row['division']:20} | {row['role']:8} | {row['type']:8} | {row['count_dancer']:,} dancers")
        
        logging.info(f"Aggregated divisional structure (dominate): {final_count} records")
    
except FileNotFoundError as e:
    print(f"\n❌ Error: File not found - {e}")
    print(f"   💡 Make sure 'changed_dancer_role_info.csv' exists")
    logging.error(f"File not found for divisional structure aggregation: {e}")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    logging.error(f"Error aggregating divisional structure (dominate): {e}")
    import traceback
    traceback.print_exc()

print(f"\n✅ Cell 38 complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# POST-PROCESS CSV FILES - CRITICAL FIXES
# ═══════════════════════════════════════════════════════════════════
#
# 🎯 НАЗНАЧЕНИЕ: Автоматическое применение критических исправлений
#                к CSV-файлам после их создания/обновления
#
# 📋 ИСПРАВЛЕНИЯ:
#    1. events_wsdc.csv - добавление event_instance_id (уникальный ключ)
#    2. events_wsdc.csv - удаление записей без location
#    3. location_info.csv - добавление country для Singapore
#
# ⚠️  ИСПОЛЬЗОВАНИЕ:
#    Запускать ПОСЛЕ создания/обновления events_wsdc.csv и location_info.csv
#    Или как финальный шаг перед использованием в Tableau
# ═══════════════════════════════════════════════════════════════════

import os
import logging

print("\n" + "="*80)
print("🔧 POST-PROCESSING CSV FILES (CRITICAL FIXES)")
print("="*80 + "\n")

files_processed = 0
files_skipped = 0

# ═══════════════════════════════════════════════════════════════════
# 1. events_wsdc.csv - Добавить event_instance_id
# ═══════════════════════════════════════════════════════════════════

if os.path.exists('events_wsdc.csv'):
    print("1️⃣  Processing events_wsdc.csv...")
    
    try:
        df_events = pd.read_csv('events_wsdc.csv')
        initial_count = len(df_events)
        
        # Проверяем, есть ли уже event_instance_id
        if 'event_instance_id' not in df_events.columns:
            print(f"   ✅ Adding event_instance_id...")
            
            # Добавляем уникальный ключ
            df_events['event_instance_id'] = range(1, len(df_events) + 1)
            
            # Переупорядочиваем колонки
            columns = df_events.columns.tolist()
            if 'id' in columns:
                # event_instance_id первым, затем остальные
                other_cols = [c for c in columns if c != 'event_instance_id']
                df_events = df_events[['event_instance_id'] + other_cols]
            
            print(f"      • Added event_instance_id: 1 → {len(df_events):,}")
        else:
            print(f"   ℹ️  event_instance_id already exists")
        
        # ═══════════════════════════════════════════════════════════════════
        # DATA QUALITY IMPROVEMENTS - EVENTS_WSDC
        # ═══════════════════════════════════════════════════════════════════
        
        if DATA_PREPROCESSING_AVAILABLE:
            print(f"\n   🔄 Applying data quality improvements to events_wsdc...")
            
            try:
                # 1. Нормализация дат
                print(f"      📅 Normalizing dates...")
                df_events = normalize_dates(df_events)
                
                # 2. Статистика улучшений
                if 'parsed_date' in df_events.columns:
                    parsed_dates = df_events['parsed_date'].notna().sum()
                    print(f"      ✅ Dates normalized:")
                    print(f"         • Parsed dates: {parsed_dates}/{len(df_events)} ({parsed_dates/len(df_events)*100:.1f}%)")
                
                if 'event_year' in df_events.columns:
                    years_extracted = df_events['event_year'].notna().sum()
                    print(f"         • Years extracted: {years_extracted}/{len(df_events)} ({years_extracted/len(df_events)*100:.1f}%)")
                
                if 'event_month' in df_events.columns:
                    months_extracted = df_events['event_month'].notna().sum()
                    print(f"         • Months extracted: {months_extracted}/{len(df_events)} ({months_extracted/len(df_events)*100:.1f}%)")
                
            except Exception as e:
                print(f"      ⚠️  Warning: Data quality improvements failed: {e}")
                logging.warning(f"Data quality improvements failed: {e}")
        else:
            print(f"   ℹ️  Data preprocessing not available, skipping quality improvements")
        
        # Удаляем записи без location
        null_location = df_events['location'].isnull().sum()
        if null_location > 0:
            print(f"   ⚠️  Removing {null_location} events without location...")
            df_events = df_events.dropna(subset=['location'])
            removed = initial_count - len(df_events)
            print(f"      • Removed {removed} events")
        
        # Сохраняем
        df_events.to_csv('events_wsdc.csv', index=False)
        
        print(f"   ✅ events_wsdc.csv updated!")
        print(f"      • Total events: {len(df_events):,}")
        print(f"      • event_instance_id unique: {df_events['event_instance_id'].is_unique}")
        
        files_processed += 1
        logging.info(f"Post-processed events_wsdc.csv: {len(df_events):,} events")
        
    except Exception as e:
        print(f"   ❌ Error processing events_wsdc.csv: {e}")
        logging.error(f"Error in events_wsdc post-processing: {e}")
        files_skipped += 1
else:
    print("1️⃣  events_wsdc.csv not found - skipping")
    files_skipped += 1

# ═══════════════════════════════════════════════════════════════════
# 2. location_info.csv - Добавить country для Singapore
# ═══════════════════════════════════════════════════════════════════

print("\n2️⃣  Processing location_info.csv...")

if 'location_info' in globals() and isinstance(location_info, pd.DataFrame):
    print(f"   ✅ Using location_info from memory...")
    df_location = location_info.copy()
    source = "memory"
elif os.path.exists('location_info.csv'):
    print(f"   ✅ Loading location_info.csv...")
    df_location = pd.read_csv('location_info.csv')
    source = "file"
else:
    print(f"   ⚠️  location_info not found - skipping")
    df_location = None
    files_skipped += 1

if df_location is not None:
    try:
        # Находим записи без country
        null_country = df_location['event_country'].isnull().sum()
        
        if null_country > 0:
            print(f"   ⚠️  Found {null_country} locations without country")
            
            # Исправляем Singapore
            singapore_mask = (
                (df_location['event_city'] == 'Singapore') & 
                (df_location['event_country'].isnull())
            )
            
            affected = singapore_mask.sum()
            
            if affected > 0:
                df_location.loc[singapore_mask, 'event_country'] = 'Singapore'
                print(f"      • Fixed Singapore: {affected} locations")
            
            # ═══════════════════════════════════════════════════════════════════
            # DATA QUALITY IMPROVEMENTS - LOCATION_INFO
            # ═══════════════════════════════════════════════════════════════════
            
            if DATA_PREPROCESSING_AVAILABLE:
                print(f"\n   🔄 Applying data quality improvements to location_info...")
                
                try:
                    # 1. Нормализация географии
                    print(f"      📍 Normalizing geography...")
                    df_location = normalize_geography(df_location)
                    
                    # 2. Статистика улучшений
                    if 'event_country' in df_location.columns:
                        standardized_count = df_location['event_country'].notna().sum()
                        print(f"      ✅ Geography normalized:")
                        print(f"         • Countries filled: {standardized_count}/{len(df_location)} ({standardized_count/len(df_location)*100:.1f}%)")
                    
                    if 'coordinates_valid' in df_location.columns:
                        valid_coords = df_location['coordinates_valid'].sum()
                        print(f"         • Valid coordinates: {valid_coords}/{len(df_location)} ({valid_coords/len(df_location)*100:.1f}%)")
                    
                    if 'event_location_standardized' in df_location.columns:
                        standardized_locs = df_location['event_location_standardized'].notna().sum()
                        print(f"         • Standardized locations: {standardized_locs}/{len(df_location)} ({standardized_locs/len(df_location)*100:.1f}%)")
                    
                    if 'event_state' in df_location.columns:
                        states_filled = df_location['event_state'].notna().sum()
                        print(f"         • States filled: {states_filled}/{len(df_location)} ({states_filled/len(df_location)*100:.1f}%)")
                    
                except Exception as e:
                    print(f"      ⚠️  Warning: Data quality improvements failed: {e}")
                    logging.warning(f"Data quality improvements failed: {e}")
            else:
                print(f"   ℹ️  Data preprocessing not available, skipping quality improvements")
            
            # Проверка: остались ли записи без country?
            null_country_after = df_location['event_country'].isnull().sum()
            if null_country_after > 0:
                print(f"      ⚠️  Warning: {null_country_after} locations still without country")
                # Выводим их для диагностики
                missing = df_location[df_location['event_country'].isnull()][['location_id', 'event_city', 'event_location']].head(5)
                for idx, row in missing.iterrows():
                    print(f"         location_id {row['location_id']}: {row['event_city']} ({row['event_location']})")
            else:
                print(f"      ✅ All locations now have country!")
        else:
            print(f"   ✅ All locations already have country")
        
        # Сохраняем
        df_location.to_csv('location_info.csv', index=False)
        
        # Обновляем в памяти, если была оттуда
        if source == "memory":
            location_info = df_location
        
        print(f"   ✅ location_info.csv updated!")
        print(f"      • Total locations: {len(df_location):,}")
        print(f"      • country completeness: {df_location['event_country'].notna().sum()}/{len(df_location)} ({df_location['event_country'].notna().sum()/len(df_location)*100:.1f}%)")
        
        files_processed += 1
        logging.info(f"Post-processed location_info.csv: {len(df_location):,} locations")
        
    except Exception as e:
        print(f"   ❌ Error processing location_info.csv: {e}")
        logging.error(f"Error in location_info post-processing: {e}")
        files_skipped += 1

# ═══════════════════════════════════════════════════════════════════
# ФИНАЛЬНЫЙ ОТЧЕТ
# ═══════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📊 POST-PROCESSING SUMMARY")
print("="*80 + "\n")
print(f"   ✅ Files processed: {files_processed}")
print(f"   ⚠️  Files skipped: {files_skipped}")

if files_processed > 0:
    print(f"\n   💡 Changes applied:")
    print(f"      • events_wsdc.csv: event_instance_id added (if missing)")
    print(f"      • events_wsdc.csv: records without location removed")
    print(f"      • location_info.csv: Singapore country added (if missing)")
    print(f"\n   ✅ CSV files are now ready for Tableau!")
else:
    print(f"\n   ℹ️  No files were processed")

print(f"\n✅ Cell 39 (Post-processing) complete!")



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DATA QUALITY VALIDATION
# ═══════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("🔍 DATA QUALITY VALIDATION")
print("="*80 + "\n")

if DATA_PREPROCESSING_AVAILABLE:
    try:
        # Собираем все данные для валидации
        validation_data = {}
        
        if 'location_info' in globals() and isinstance(location_info, pd.DataFrame):
            validation_data['location_info'] = location_info
        if 'events_wsdc' in globals() and isinstance(events_wsdc, pd.DataFrame):
            validation_data['events_wsdc'] = events_wsdc
        if 'dancer_role_info' in globals() and isinstance(dancer_role_info, pd.DataFrame):
            validation_data['dancer_role_info'] = dancer_role_info
        if 'dancers_results_info' in globals() and isinstance(dancers_results_info, pd.DataFrame):
            validation_data['dancers_results_info'] = dancers_results_info
        
        if validation_data:
            # Выполняем валидацию
            issues = validate_data_quality(validation_data)
            
            if issues:
                print(f"\n⚠️  Found {len(issues)} data quality issues:\n")
                
                # Группируем по severity
                critical = [i for i in issues if i.get('severity') == 'CRITICAL']
                high = [i for i in issues if i.get('severity') == 'HIGH']
                medium = [i for i in issues if i.get('severity') == 'MEDIUM']
                low = [i for i in issues if i.get('severity') == 'LOW']
                
                if critical:
                    print("🔴 CRITICAL ISSUES:")
                    for issue in critical:
                        print(f"   • {issue['table']}.{issue['field']}: {issue['issue']}")
                        if 'examples' in issue:
                            print(f"     Examples: {issue['examples'][:3]}")
                    print()
                
                if high:
                    print("🟠 HIGH PRIORITY ISSUES:")
                    for issue in high:
                        print(f"   • {issue['table']}.{issue['field']}: {issue['issue']}")
                        if 'examples' in issue:
                            print(f"     Examples: {issue['examples'][:3]}")
                    print()
                
                if medium:
                    print("🟡 MEDIUM PRIORITY ISSUES:")
                    for issue in medium:
                        print(f"   • {issue['table']}.{issue['field']}: {issue['issue']}")
                    print()
                
                if low:
                    print("🟢 LOW PRIORITY ISSUES:")
                    for issue in low:
                        print(f"   • {issue['table']}.{issue['field']}: {issue['issue']}")
                    print()
                
                # Логируем проблемы
                for issue in issues:
                    logging.warning(f"Data quality issue: {issue['table']}.{issue['field']} - {issue['issue']}")
            else:
                print("✅ All data quality checks passed!")
                print("   No issues found in the data")
        else:
            print("ℹ️  No data available for validation")
            
    except Exception as e:
        print(f"❌ Error during validation: {e}")
        logging.error(f"Data quality validation failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("ℹ️  Data preprocessing not available, skipping validation")

print("\n✅ Data Quality Validation complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MERGE HISTORICAL AND CURRENT DANCER_ROLE_INFO
# ═══════════════════════════════════════════════════════════════════
"""
Объединяет историческую версию changed_dancer_role_info.csv 
с текущим snapshot dancer_role_info.csv.

Процесс:
1. Загрузить историческую версию из changed_dancer_role_info.csv
2. Загрузить текущий snapshot из dancer_role_info.csv
3. Сравнить и найти изменения (новые/измененные танцоры)
4. Добавить изменения с сегодняшней датой
5. Сохранить объединенный результат

Результат:
- changed_dancer_role_info.csv обновлен с новой историей
- Сохранена преемственность дат
"""

from datetime import datetime

print("\n" + "="*80)
print("🔄 ОБЪЕДИНЕНИЕ ИСТОРИЧЕСКИХ И ТЕКУЩИХ ДАННЫХ")
print("="*80 + "\n")

try:
    # 1. Загружаем историческую версию
    print("1️⃣  Загрузка исторической версии...")
    
    if not os.path.exists('changed_dancer_role_info.csv'):
        print("   ⚠️  changed_dancer_role_info.csv не найден")
        print("   💡 Создаем новую историю из текущих данных...")
        
        if 'dancer_role_info' in globals() and isinstance(dancer_role_info, pd.DataFrame):
            df_history = pd.DataFrame()
            df_current = dancer_role_info.copy()
        else:
            raise ValueError("dancer_role_info не найден в памяти")
    else:
        df_history = pd.read_csv('changed_dancer_role_info.csv', low_memory=False)
        print(f"   ✅ Загружено: {len(df_history):,} записей")
        
        if 'update_date' in df_history.columns:
            print(f"      📅 Уникальных дат: {df_history['update_date'].nunique()}")
            dates = sorted(df_history['update_date'].unique())
            print(f"      📆 Диапазон: {dates[0]} → {dates[-1]}")
    
    # 2. Загружаем текущий snapshot
    print("\n2️⃣  Загрузка текущего snapshot...")
    
    if 'dancer_role_info' in globals() and isinstance(dancer_role_info, pd.DataFrame):
        df_current = dancer_role_info.copy()
        print(f"   ✅ Загружено из памяти: {len(df_current):,} записей")
    elif os.path.exists('dancer_role_info.csv'):
        df_current = pd.read_csv('dancer_role_info.csv')
        print(f"   ✅ Загружено из файла: {len(df_current):,} записей")
    else:
        raise ValueError("dancer_role_info не найден")
    
    # 3. Сравниваем и находим изменения
    print("\n3️⃣  Поиск изменений...")
    
    if len(df_history) == 0:
        # Первый запуск - все записи новые
        print(f"   🆕 Первый запуск - все {len(df_current):,} записей будут добавлены")
        all_changed_ids = df_current['dancer_id'].tolist()
    else:
        # Получаем последнюю дату и последний snapshot
        last_date = df_history['update_date'].max()
        print(f"   📅 Последняя дата в истории: {last_date}")
        
        df_last_snapshot = df_history[df_history['update_date'] == last_date].copy()
        df_last_snapshot = df_last_snapshot.drop(columns=['update_date'])
        print(f"   📊 Последний snapshot: {len(df_last_snapshot):,} записей")
        
        # Приводим колонки к одному порядку
        common_cols = [col for col in df_current.columns if col in df_last_snapshot.columns]
        df_current_sorted = df_current[common_cols].sort_values('dancer_id').reset_index(drop=True)
        df_last_sorted = df_last_snapshot[common_cols].sort_values('dancer_id').reset_index(drop=True)
        
        # Находим новые и измененные записи
        merged = df_current_sorted.merge(
            df_last_sorted,
            on='dancer_id',
            how='left',
            suffixes=('_current', '_last'),
            indicator=True
        )
        
        # Новые танцоры
        new_dancers = merged[merged['_merge'] == 'left_only']['dancer_id'].unique()
        
        # Измененные танцоры (проверяем все колонки)
        changed_dancers = []
        for col in common_cols:
            if col != 'dancer_id':
                col_current = f"{col}_current"
                col_last = f"{col}_last"
                if col_current in merged.columns and col_last in merged.columns:
                    mask = (merged[col_current] != merged[col_last]) & \
                           ~(merged[col_current].isna() & merged[col_last].isna())
                    changed_dancers.extend(merged[mask]['dancer_id'].unique())
        
        changed_dancers = list(set(changed_dancers))
        all_changed_ids = list(set(list(new_dancers) + changed_dancers))
        
        print(f"   🆕 Новых танцоров: {len(new_dancers):,}")
        print(f"   🔄 Измененных записей: {len(changed_dancers):,}")
    
    # 4. Формируем новые записи
    if all_changed_ids:
        today = datetime.now().strftime('%Y-%m-%d')
        df_changes = df_current[df_current['dancer_id'].isin(all_changed_ids)].copy()
        df_changes['update_date'] = today
        
        print(f"\n4️⃣  Подготовка новых записей...")
        print(f"   📝 Записей для добавления: {len(df_changes):,}")
        print(f"   📅 Дата: {today}")
        
        # 5. Объединяем с историей
        print("\n5️⃣  Объединение с историей...")
        
        # Приводим порядок колонок
        if len(df_history) > 0:
            cols_order = list(df_history.columns)
            df_changes = df_changes[cols_order]
            df_merged = pd.concat([df_history, df_changes], ignore_index=True)
        else:
            df_merged = df_changes
        
        print(f"   ✅ Объединено:")
        print(f"      • История: {len(df_history):,} записей")
        print(f"      • Новые: {len(df_changes):,} записей")
        print(f"      • ИТОГО: {len(df_merged):,} записей")
        
        # 6. Сохраняем результат
        print("\n6️⃣  Сохранение результата...")
        df_merged.to_csv('changed_dancer_role_info.csv', index=False)
        print(f"   ✅ Сохранено в changed_dancer_role_info.csv")
        
        # Статистика
        print(f"\n📊 ИТОГОВАЯ СТАТИСТИКА:")
        print(f"   📝 Всего записей: {len(df_merged):,}")
        print(f"   📅 Уникальных дат: {df_merged['update_date'].nunique()}")
        print(f"   👤 Уникальных танцоров: {df_merged['dancer_id'].nunique()}")
        
        dates = sorted(df_merged['update_date'].unique())
        print(f"\n   📆 Диапазон дат: {dates[0]} → {dates[-1]}")
        
        if len(dates) <= 10:
            print(f"\n   📋 Записи по датам:")
            for date in dates:
                count = (df_merged['update_date'] == date).sum()
                print(f"      • {date}: {count:,} записей")
        else:
            print(f"\n   📋 Последние 5 дат:")
            for date in dates[-5:]:
                count = (df_merged['update_date'] == date).sum()
                print(f"      • {date}: {count:,} записей")
        
        logging.info(f"Merged historical and current dancer_role_info: {len(df_merged):,} total records")
        
    else:
        print(f"\n✅ НЕТ ИЗМЕНЕНИЙ!")
        if len(df_history) > 0:
            last_date = df_history['update_date'].max()
            print(f"   Текущие данные идентичны последнему snapshot от {last_date}")
            print(f"   Файл changed_dancer_role_info.csv не изменен")
        
        logging.info("No changes detected in dancer_role_info")

except Exception as e:
    print(f"\n❌ ОШИБКА: {e}")
    logging.error(f"Error merging dancer_role_info history: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*80)
print("✅ Cell 40 (Merge History) complete!")
print("="*80 + "\n")


In [ ]:
# ============================================================================
# 📤 GITHUB LFS AUTO-PUSH
# Автоматическая загрузка обновленных CSV в telegram-news-bot
# ============================================================================

print("\n" + "=" * 80)
print("📤 PUSHING UPDATED CSV FILES TO GITHUB LFS")
print("=" * 80 + "\n")

import subprocess
import shutil
from pathlib import Path
from datetime import datetime
import os

# === КОНФИГУРАЦИЯ ===
SOURCE_DIR = Path('/Users/ania/.cursor/projects/tableau/My-Tableau-Projects/WSDC/WSDC Points')
BOT_REPO = Path('/Users/ania/.cursor/projects/python/telegram-news-bot')
DATA_DIR = BOT_REPO / 'data'

# CSV файлы для копирования
CSV_FILES = [
    'dancers_results_info.csv',
    'dancers_points_info.csv',
    'dancer_role_info.csv',
    'location_info.csv',
    'events_wsdc.csv'
]

# === КОПИРОВАНИЕ ФАЙЛОВ ===
print("📁 Step 1: Copying CSV files to bot repo...")
print(f"   Source: {SOURCE_DIR}")
print(f"   Destination: {DATA_DIR}")
print()

# Создаем data/ если не существует
DATA_DIR.mkdir(parents=True, exist_ok=True)

copied_files = []
for csv_file in CSV_FILES:
    source = SOURCE_DIR / csv_file
    dest = DATA_DIR / csv_file
    
    if source.exists():
        shutil.copy2(source, dest)
        size_mb = source.stat().st_size / (1024 * 1024)
        print(f"   ✅ {csv_file:<35} ({size_mb:>6.2f} MB)")
        copied_files.append(csv_file)
    else:
        print(f"   ⚠️  {csv_file:<35} - FILE NOT FOUND")

print(f"\n   📊 Copied {len(copied_files)}/{len(CSV_FILES)} files")
print()

if not copied_files:
    print("❌ No files copied! Aborting.")
else:
    # === GIT OPERATIONS ===
    print("🔀 Step 2: Git commit and push...")
    os.chdir(BOT_REPO)
    
    try:
        # Проверяем статус
        status_result = subprocess.run(
            ['git', 'status', '--porcelain', 'data/'],
            capture_output=True,
            text=True,
            check=True
        )
        
        if not status_result.stdout.strip():
            print("   ℹ️  No changes detected in CSV files")
            print("   ✅ CSVs are already up-to-date!")
        else:
            # Add files
            subprocess.run(['git', 'add', 'data/'], check=True)
            print("   ✅ Files staged")
            
            # Commit
            commit_msg = f'Update WSDC points data - {datetime.now().strftime("%Y-%m-%d %H:%M")}'
            subprocess.run(
                ['git', 'commit', '-m', commit_msg],
                check=True,
                capture_output=True
            )
            print(f"   ✅ Committed: {commit_msg}")
            
            # Push
            print("   📤 Pushing to GitHub (may take a while for LFS)...")
            subprocess.run(['git', 'push', 'origin', 'main'], check=True, capture_output=True, timeout=300)
            print("   ✅ Pushed to GitHub!")
            
            print()
            print("=" * 80)
            print("✅ CSV FILES SUCCESSFULLY UPLOADED TO GITHUB LFS!")
            print("=" * 80)
            print()
            print("🎯 NEXT STEP: Run Results Bot on GitHub")
            print("   https://github.com/zlat1109/wsdc-telegram-bot/actions")
    
    except subprocess.CalledProcessError as e:
        print(f"   ❌ Git error: {e}")
        print("\n   💡 Run manually: cd to telegram-news-bot and run 'git push'")
    
    except Exception as e:
        print(f"   ❌ Error: {e}")
